In [ ]:
# All from: https://gitlab.uni-mannheim.de/jpmac/bpdp

In [ ]:
# We import the neccessary packages in the beginning
import os
import math
from statistics import mean,stdev
import pm4py
from pm4py.objects.conversion.log import converter as log_converter
from pm4py.objects.conversion.bpmn import converter as bpmn_converter
from sklearn.impute import SimpleImputer
import copy
import numpy as np
import pandas as pd
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from imblearn.under_sampling import OneSidedSelection
from sklearn.preprocessing import StandardScaler    
from sklearn.model_selection import train_test_split
import sklearn
import tqdm
import time

In [ ]:
# Returns a path to the file selected by the user
# Input: The folder in which to look for the files - the default is the current folder
def ask_for_path(rel_path='', index = -1):
    #Crawl all files in the input folder
    print("The following files are available in the input folder:\n")

    count = 0
    # file_list = os.listdir(os.getcwd() + rel_path)
    file_list = os.listdir(rel_path)
    for file in file_list:
        print(str(count) + " - " + file)
        count+=1

    if(index == -1):
        #Ask for which of the files shall be transformed and select it.
        inp = input("Please choose from the list above which of the files shall be transformed by typing the corresponding number.")
    else:
        #Automatic iteration
        print('Automatic Iteration.')
        inp = index

    input_file = file_list[int(inp)]

    # return (os.getcwd() + rel_path + input_file)
    return (rel_path + input_file)

In [ ]:
# this is a help function to print petri nets
def output_petri_net(net, initial_marking, final_marking, file_name, label):
    #init visualizer
    parameters = {pn_visualizer.Variants.FREQUENCY.value.Parameters.FORMAT: OUTPUT_FORMAT, 'label':'The Round Table'}   #Add frequency to graph
    gviz = pn_visualizer.apply(net, initial_marking, final_marking, parameters=parameters, variant=pn_visualizer.Variants.FREQUENCY, log=log)

    gviz.attr(label=label)
    pn_visualizer.save(gviz, os.getcwd() + REL_OUTPUT_PATH + file_name + "." + OUTPUT_FORMAT)

In [ ]:
def get_output_path(file_name,REL_OUTPUT_PATH = "/Output Tree/"):
    # return (os.getcwd() + REL_OUTPUT_PATH + file_name)
    return (REL_OUTPUT_PATH + file_name)

In [ ]:
# this function converts a selected file in the path that is the input into a log
def transform_to_log(file_path):
    filename, file_extension = os.path.splitext(file_path)
    x,z =os.path.split(file_path)
    
    if file_extension == '.csv':
        log_csv = pd.read_csv(file_path,sep=None,encoding='utf-8-sig')
        
        """ not used in my comparison
        if z =='mobis_challenge_log_2019.csv' or z =='mobis_challenge_log_2019_only_complete_cases.csv':
            log_csv['end'] = pd.to_datetime(log_csv['end'])
            log_csv['start'] = pd.to_datetime(log_csv['start'])
            log_csv['cost'] = log_csv['cost'].apply(pd.to_numeric, errors='coerce')
            log_csv.rename(columns={'cost': 'case:cost','case':'case:concept:name','activity':'concept:name','end':'time:timestamp', 'user':'org:resource'}, inplace=True)
        
        elif z =='mobis_challenge_log_2019_original.csv':
            log_csv['end'] = pd.to_datetime(log_csv['end'])
            log_csv['start'] = pd.to_datetime(log_csv['start'])
            log_csv['cost'] = log_csv['cost'].apply(pd.to_numeric, errors='coerce')
            log_csv.rename(columns={'case':'case:concept:name','activity':'concept:name','start':'time:timestamp', 'user':'org:resource'}, inplace=True)
        """
        
        log_csv['time:timestamp'] = pd.to_datetime(log_csv['time:timestamp'], format='mixed')
        log = log_converter.apply(log_csv)

    elif file_extension == '.xes':
        log = pm4py.read_xes(file_path)
        log = pm4py.convert_to_event_log(log)
    elif file_extension == '.dfg':
        log = pm4py.read_dfg(file_path)
    else:
        print("Current filetype is equal to {}. \nPlease input a file with any of the following extensions: - csv; - xes; - dfg".format(str(file_extension)))
        return -1

    return log

In [ ]:
def get_all_activities_from_log(log):
    activities=[]
    for trace in log:
        for event in trace:
            if activities.count(event['concept:name'])==0:
                activities.append(event['concept:name'])
    return activities

In [ ]:
# this function enriches each trace by the event 1...m, resource 1...m, Weekday start and end attributes until a given prefix length
def complex_index_encoding(log, pref_length=5):
    max_ev=0
    for trace in log:
        i=0
        for event in trace:
            i+=1
        if i>max_ev:
            max_ev=i
    
    if pref_length > max_ev:
        print('The prefix length is larger than the maximum trace length; Maximum trace length will be used.')
        pref_length = max_ev

    #weekdays
    weekDaysMapping = ("Monday", "Tuesday",
                    "Wednesday", "Thursday",
                    "Friday", "Saturday",
                    "Sunday")

    for trace in log:
        for event in trace:
            trace.attributes['weekday_start']=weekDaysMapping[event['time:timestamp'].weekday()]
            break
    if pref_length == max_ev:
        for trace in log:
            for event in trace:
                trace.attributes['weekday_end']=weekDaysMapping[event['time:timestamp'].weekday()]
    
    j=0
    no_evs={}
    for trace in log:
        i=0
        for event in trace:
            i+=1
            if i==1:
                st_time=event['time:timestamp'].day+event['time:timestamp'].hour/24+event['time:timestamp'].minute/(24*60)
            if i<= pref_length:
                trace.attributes['event_'+str(i)]=event['concept:name']
                trace.attributes['resource_'+str(i)]=str(event['org:resource'])
                trace.attributes['month_'+str(i)]=(str(event['time:timestamp'].month)+'_'+str(event['time:timestamp'].year))
                #trace.attributes['elapsed_time']=event['time:timestamp'].day+event['time:timestamp'].hour/24+event['time:timestamp'].minute/(24*60)-st_time
        no_evs[j]=i
        j+=1

    j=0
    for trace in log:
        if no_evs[j]<max_ev:
            fill=no_evs[j]+1
            for k in range(fill,max(max_ev,pref_length)+1):
                trace.attributes['event_'+str(k)]=np.nan
                trace.attributes['resource_'+str(k)]=np.nan

    return log

In [ ]:
path_event_log = "../../../../../data/data/"
path_process_model = "../../../data/process_models/"

!ls ../../../../../data/data/
!ls ../../../data/process_models/

In [ ]:
##########
"""Settings"""
##########

# set the input and output path according to the files you want to select

REL_INPUT_PATH_LOG = path_event_log # here lie the event logs (.csv), the to-be model (.bpmn) and the already aligned traces (.pkl)
REL_INPUT_PATH_MODEL = path_process_model

REL_OUTPUT_PATH = 'aligned_traces/'

OUTPUT_FORMAT = "png"

In [ ]:
# generate the log from the input path
file= ask_for_path(REL_INPUT_PATH_LOG, 6) # adjust to your path

log=transform_to_log(file)

ref_log=transform_to_log(file)

In [ ]:
print(ref_log)

In [ ]:
file= ask_for_path(REL_INPUT_PATH_MODEL,9)# adjust to your path

bpmn_graph = pm4py.read_bpmn(file)
#pm4py.write_bpmn(bpmn_graph, "ru.bpmn", enable_layout=True)

net, initial_marking, final_marking = bpmn_converter.apply(bpmn_graph)
#net, initial_marking, final_marking=pm4py.read_pnml(file)
# pm4py.visualization.petri_net.visualizer(net, initial_marking, final_marking)
# output_petri_net(net, initial_marking, final_marking,'Basis_PN', 'test')
pm4py.view_petri_net(net, initial_marking, final_marking)

In [ ]:
def generate_alignments_pkl(log, net, initial_marking, final_marking):
    aligned_traces = pm4py.conformance_diagnostics_alignments(log, net, initial_marking, final_marking)
    i=0
    dev=[]
    for trace in log:
        no_moves=len(aligned_traces[i]['alignment'])
        for j in range(0,len(aligned_traces[i]['alignment'])):
            if aligned_traces[i]['alignment'][j][1] == None or aligned_traces[i]['alignment'][j][0]==aligned_traces[i]['alignment'][j][1]:
                next
            else:
                if not str(aligned_traces[i]['alignment'][j]) in dev:
                    dev.append(str(aligned_traces[i]['alignment'][j]))
                #trace.attributes[str(aligned_traces[i]['alignment'][j])]=1
        i+=1

    # f = open('aligned_traces/aligned_traces_binet_12A.pkl','wb')
    f = open('aligned_traces/aligned_traces_20dom.pkl','wb')
    pickle.dump(aligned_traces,f)
    f.close()
    return dev

In [ ]:
# dev, aligned_traces=generate_alignments_pkl(log, net, initial_marking, final_marking)
dev=generate_alignments_pkl(log, net, initial_marking, final_marking)
print(len(dev))
dev

In [ ]:
file= ask_for_path('aligned_traces/', 0)# adjust to your path
with open(file, 'rb') as f:
     aligned_traces=pickle.load(f)

In [ ]:
## train data
class TrainData(Dataset):
    
    def __init__(self, X_data, y_data):
        self.X_data = X_data
        self.y_data = y_data
        
    def __getitem__(self, index):
        return self.X_data[index], self.y_data[index]
        
    def __len__ (self):
        return len(self.X_data)

## test data    
class TestData(Dataset):
    
    def __init__(self, X_data):
        self.X_data = X_data

    def __getitem__(self, index):
        return self.X_data[index]
        
    def __len__ (self):
        return len(self.X_data)

In [ ]:
# we define our FFN: sepearate
class BinaryClassificationIndiv(nn.Module):
    def __init__(self, no_columns):
        super(BinaryClassificationIndiv, self).__init__()
        # Number of input features is 12.
        self.layer_1 = nn.Linear(no_columns, 256)
        self.activation1 = nn.LeakyReLU()
        self.layer_2 = nn.Linear(256, 256)
        self.activation2 = nn.LeakyReLU()
        self.layer_out = nn.Linear(256, 2)
        
        
        self.dropout = nn.Dropout(p=0.1)
        self.batchnorm1 = nn.LayerNorm(256)
        self.batchnorm2 = nn.LayerNorm(256)
        self.Softmax = nn.Softmax()
        
    def forward(self, inputs):
        x = self.activation1(self.layer_1(inputs))
        x = self.batchnorm1(x)
        x = self.activation2(self.layer_2(x))
        x = self.batchnorm2(x)
        x = self.dropout(x)
        x = self.layer_out(x)
        
        return x

In [ ]:
# we define our FFN collective
class BinaryClassification(nn.Module):
    def __init__(self, no_columns, no_devs):
        super(BinaryClassification, self).__init__()
        # Number of input features is 12.
        self.layer_1 = nn.Linear(no_columns, 2048)
        self.activation1 = nn.LeakyReLU()
        self.layer_2 = nn.Linear(2048, 2048)
        self.activation2 = nn.LeakyReLU()
        self.layer_3 = nn.Linear(2048, 1024)
        self.activation3 = nn.LeakyReLU()
        self.layer_out = nn.Linear(1024, no_devs)


        self.dropout = nn.Dropout(p=0.1)
        self.batchnorm1 = nn.LayerNorm(2048)
        self.batchnorm2 = nn.LayerNorm(1024)
        self.Sigmoid = nn.Sigmoid()

    def forward(self, inputs):
        x = self.activation1(self.layer_1(inputs))
        x = self.batchnorm1(x)
        x = self.activation2(self.layer_2(x))
        x = self.batchnorm1(x)
        x = self.activation3(self.layer_3(x))
        x = self.batchnorm2(x)
        x = self.dropout(x)
        x = self.layer_out(x)

        return x

In [ ]:
class BPDP_LSTM(nn.Module):
    def __init__(self, vocab_events, vocab_resources, no_TA, vocab_month):
        super(BPDP_LSTM, self).__init__()
        self.embedding_e = nn.Embedding(vocab_events, 16) # hier auf 8 / 16
        self.activation1 = nn.LeakyReLU()
        self.lstm_e = nn.LSTM(input_size=16, hidden_size=64, num_layers=1, batch_first=True, dropout=0.1)
        self.linear_e = nn.Linear(64, 32)
        self.embedding_r = nn.Embedding(vocab_resources, 16)
        self.lstm_r = nn.LSTM(input_size=16, hidden_size=64, num_layers=1, batch_first=True, dropout=0.1)
        self.linear_r = nn.Linear(64, 32)
        self.embedding_m = nn.Embedding(vocab_month, 16)
        self.lstm_m = nn.LSTM(input_size=16, hidden_size=64, num_layers=1, batch_first=True, dropout=0.1)
        self.linear_m = nn.Linear(64, 32)
        self.linear_ta = nn.Linear(no_TA, 32)
        self.dropout = nn.Dropout(p=0.1)
        self.batchnorm1 = nn.LayerNorm(128)
        self.linear = nn.Linear(128, 2)
    def forward(self, evs, rs,tas, ms):
        evs= self.embedding_e(evs)
        evs, _ = self.lstm_e(evs)
        evs=self.linear_e(evs)
        evs=evs[:, -1, :]
        evs=self.activation1(evs)
        rs= self.embedding_r(rs)
        rs, _ = self.lstm_r(rs)
        rs=rs[:, -1, :]
        rs=self.activation1(rs)
        rs=self.linear_r(rs)
        ms= self.embedding_m(ms)
        ms, _ = self.lstm_m(ms)
        ms=ms[:, -1, :]
        ms=self.activation1(ms)
        ms=self.linear_m(ms)
        tas= self.linear_ta(tas)
        fin=torch.cat((evs,rs),dim=1)
        fin=torch.cat((fin,ms),dim=1)
        fin=torch.cat((fin,tas),dim=1)
        fin=self.batchnorm1(fin)
        #fin = self.dropout(fin)
        fin = self.linear(fin)
        return fin

In [ ]:
class LargerBinaryClassificationIndiv(nn.Module):
    def __init__(self, no_columns):
        super(LargerBinaryClassificationIndiv, self).__init__()
        self.layer_1 = nn.Linear(no_columns, 512)
        self.activation1 = nn.LeakyReLU()
        self.layer_2 = nn.Linear(512, 256)
        self.activation2 = nn.LeakyReLU()
        self.layer_3 = nn.Linear(256, 256)
        self.activation3 = nn.LeakyReLU()
        self.layer_out = nn.Linear(256, 2)


        self.dropout = nn.Dropout(p=0.1)
        self.batchnorm1 = nn.LayerNorm(512)
        self.batchnorm2 = nn.LayerNorm(256)
        self.Softmax = nn.Softmax()

    def forward(self, inputs):
        x = self.activation1(self.layer_1(inputs))
        x = self.batchnorm1(x)
        x = self.activation2(self.layer_2(x))
        x = self.batchnorm2(x)
        x = self.activation3(self.layer_3(x))
        x = self.dropout(x)
        x = self.layer_out(x)

        return x

In [ ]:
class EarlyStopping():
  def __init__(self, patience=10, min_delta=0, restore_best_weights=True):
    self.patience = patience
    self.min_delta = min_delta
    self.restore_best_weights = restore_best_weights
    self.best_model = None
    self.best_loss = None
    self.counter = 0
    self.status = ""
    
  def __call__(self, model, val_loss):
    if self.best_loss == None:
      self.best_loss = val_loss
      self.best_model = copy.deepcopy(model)
    elif self.best_loss - val_loss > self.min_delta:
      self.best_loss = val_loss
      self.counter = 0
      self.best_model.load_state_dict(model.state_dict())
    elif self.best_loss - val_loss <= self.min_delta:
      self.counter += 1
      if self.counter >= self.patience:
        self.status = f"Stopped on {self.counter}"
        if self.restore_best_weights:
          model.load_state_dict(self.best_model.state_dict())
        return True
    self.status = f"{self.counter}/{self.patience}"
    return False

In [ ]:
# just for printing the accuracy during training - only informational
def binary_acc(y_pred, y_test):
    y_pred_tag = torch.round(torch.nn.functional.softmax(y_pred))

    correct_results_sum = (y_pred_tag == y_test).sum().float()
    acc = correct_results_sum/y_test.shape[0]
    acc = torch.round(acc * 100)
    
    return acc

In [ ]:
def IDP_separate_CIBE(log, ref_log, aligned_traces, split=1/3, u_sample=True, early_stop=True,explained=False):
    xt,z =os.path.split(file)
    import warnings
    warnings.simplefilter('ignore')
    #### get information whether deviation happened after prefix length in DF
    i=0
    dev=[] # stores all deviations that happened
    for trace in log:
        no_moves=len(aligned_traces[i]['alignment'])
        for j in range(0,len(aligned_traces[i]['alignment'])):
            if aligned_traces[i]['alignment'][j][1] == None or aligned_traces[i]['alignment'][j][0]==aligned_traces[i]['alignment'][j][1]:
                next # we do not care for simultaneous or silent moves
            else:
                if not str(aligned_traces[i]['alignment'][j]) in dev:
                    dev.append(str(aligned_traces[i]['alignment'][j]))
        i+=1

    y_cum_test={} # dict that stores label for each prefix and deviation combinations; keys are prefix length, entries are Data Frames with index = trace and columns = deviation
    dev_df=pd.DataFrame(data=0,columns=dev, index=range(len(log))) # Data Frame that stores the information whether a deviation happened for each trace on trace level
    event_order={} # dict with event sequences for each trace
    event_count={} # dict with trace length for each trace
    max_ev=0 # will be maximum trace length
    k=0
    for trace in log:
        event_order[k]=[]
        i=0
        for event in trace:
            i+=1
            event_order[k].append(event['concept:name'])
        if i>max_ev:
            max_ev=i
        event_count[k]=len(event_order[k])
        k+=1
    i=0
    for trace in log:
        no_moves=len(aligned_traces[i]['alignment'])
        for j in range(0,len(aligned_traces[i]['alignment'])):
            if aligned_traces[i]['alignment'][j][1] == None or aligned_traces[i]['alignment'][j][0]==aligned_traces[i]['alignment'][j][1]:
                next
            else:
                dev_df[str(aligned_traces[i]['alignment'][j])][i]=1
        i+=1
    for ev in range(1,max_ev+1):
        y_cum_test[ev]=dev_df.copy() # initialize each prefix length with all traces and information whether deviation happened
    for ev in range(1,max_ev+1):
        drop_idx=[]
        for trace_idx in range(len(log)):
            if event_count[trace_idx]< ev:
                drop_idx.append(trace_idx) # drop all trace labels that do not go until prefix length
        y_cum_test[ev]=y_cum_test[ev].drop(drop_idx)
    i=0
    for trace in log:
        no_moves=len(aligned_traces[i]['alignment'])
        j=no_moves-1 # iterator over moves in alignment, starting at the end
        m=len(event_order[i]) # iterator over event sequence, starting at the end
        while j >=0:
            if aligned_traces[i]['alignment'][j][1] == None: # if silent move, just go one move further to the beginning in the alignment
                j-=1
            elif aligned_traces[i]['alignment'][j][0]==aligned_traces[i]['alignment'][j][1]:
                if event_order[i][m-1]==aligned_traces[i]['alignment'][j][0]: # if synchronous move, just go one move further to the beginning in the alignment and one event forther to the beginning in the event sequence
                    j-=1
                    m-=1
            elif event_order[i][m-1]==aligned_traces[i]['alignment'][j][0]: # log move detected
                for q in range(m,max_ev+1):
                    y_cum_test[q][str(aligned_traces[i]['alignment'][j])][i]=0 # set all prefixes from the current m to the maximum prefix length in this trace to 0 because deviation happened here but not afterwards
                j-=1
                m-=1
            elif m==max_ev:
                j-=1
            else: # model move deteceted
                for q in range(m+1,max_ev+1):
                    y_cum_test[q][str(aligned_traces[i]['alignment'][j])][i]=0 # set all prefixes after the current m to the maximum prefix length in this trace to 0 because deviation happened between m and m+1 but not afterwards
                j-=1
        i+=1
    ### y_cum_test holds information whether deviation happened after prefix length


    ## ref_log will have all attributes that will be the columns for X_test and X_train
    ref_log = complex_index_encoding(ref_log, 4000) # prepare a log with the maximum length of the feature vector from CIBE to know to pad other feature vectors
    ref_dataframe1 = pm4py.convert_to_dataframe(ref_log)
    ref_dataframe=ref_dataframe1.drop_duplicates(subset=['case:concept:name'])
    ref_dataframe=ref_dataframe.filter(like='case:', axis=1)
    ref_dataframe=ref_dataframe.drop('case:concept:name', axis=1)
    ref_dataframe.columns = ref_dataframe.columns.str.replace('case:', '')
    ref_dataframe=ref_dataframe.reset_index()
    ref_raw_dat=ref_dataframe.drop('index', axis=1)
    
    ## dataset-specific preparation (i.e., redundant attributes, convertion to numeric)
    if z=='aligned_traces_12A.pkl' or z=='aligned_traces_12O.pkl' or z=='aligned_traces_12AO.pkl':
        ref_raw_dat['AMOUNT_REQ']= pd.to_numeric(ref_raw_dat['AMOUNT_REQ'])
        ref_clean_dat=ref_raw_dat.drop('REG_DATE', axis=1)
    
    elif z=='aligned_traces_20int.pkl':
            ref_clean_dat=ref_raw_dat.drop(['Permit travel permit number','DeclarationNumber','travel permit number','id','Permit ID', 'Permit id'], axis=1)
    
    elif z=='aligned_traces_20dom.pkl':
            ref_clean_dat=ref_raw_dat.drop(['DeclarationNumber','id'], axis=1)
    
    elif z=='aligned_traces_20prep.pkl':
            ref_clean_dat=ref_raw_dat.drop(['RfpNumber','Rfp_id','Permit travel permit number','Permit id'], axis=1)
    
    elif z=='aligned_traces_20RfP.pkl':
            ref_clean_dat=ref_raw_dat.drop(['RfpNumber','Rfp_id'], axis=1)
    else:
        ref_clean_dat=ref_raw_dat.copy()
    ref_enc_dat=pd.get_dummies(ref_clean_dat)


    EPOCHS = 30
    BATCH_SIZE = 128
    LEARNING_RATE=0.0001

    X_cum={}
    metrics=pd.DataFrame(data=0, columns=dev, index=['Precision', 'Recall', 'Support', 'ROC_AUC','LenTrain', 'LenTrain_beforeUS_0', 'LenTrain_beforeUS_1', 'LenTrain_afterUS_0', 'LenTrain_afterUS_1'])

    # prepare X for all prefix lengths
    for prefix in range(1,max_ev+1):
        complex_index_encoding(log,prefix)
        dataframe1 = pm4py.convert_to_dataframe(log)
        dataframe=dataframe1.drop_duplicates(subset=['case:concept:name'])
        dataframe=dataframe.filter(like='case:', axis=1)
        dataframe=dataframe.drop('case:concept:name', axis=1)
        dataframe.columns = dataframe.columns.str.replace('case:', '')
        dataframe=dataframe.reset_index()
        raw_dat=dataframe.drop('index', axis=1)
        
        if z=='aligned_traces_12A.pkl' or z=='aligned_traces_12O.pkl' or z=='aligned_traces_12AO.pkl':
            raw_dat['AMOUNT_REQ']= pd.to_numeric(raw_dat['AMOUNT_REQ'])
            clean_dat=raw_dat.drop('REG_DATE', axis=1)
        
        elif z=='aligned_traces_20int.pkl':
            clean_dat=raw_dat.drop(['Permit travel permit number','DeclarationNumber','travel permit number','id','Permit ID', 'Permit id'], axis=1)
        
        # my! delete dom
        elif z=='aligned_traces_20dom.pkl':
            clean_dat=raw_dat.drop(['DeclarationNumber','id'], axis=1)
        
        elif z=='aligned_traces_20prep.pkl':
            clean_dat=raw_dat.drop(['RfpNumber','Rfp_id','Permit travel permit number','Permit id'], axis=1)
        
        elif z=='aligned_traces_20RfP.pkl':
            clean_dat=raw_dat.drop(['RfpNumber','Rfp_id'], axis=1)
        
        else:
            clean_dat=raw_dat.copy()
        enc_dat=pd.get_dummies(clean_dat)
        for key in ref_enc_dat.columns:
            if not key in enc_dat.columns:
                enc_dat[key]=0 # pad all prefixes to maximum lengths with 0
        imp = SimpleImputer(missing_values=np.nan, strategy='constant',fill_value=0)
        enc_dat=pd.DataFrame(data=imp.fit_transform(enc_dat),columns=enc_dat.columns)

        X_cum[prefix]=enc_dat.copy()
        drop_idx=[]
        for trace_idx in range(len(log)):
            if event_count[trace_idx]< prefix:
                drop_idx.append(trace_idx)

        X_cum[prefix]=X_cum[prefix].drop(drop_idx)

    positive_weights = {}
    negative_weights = {}
    for label in dev:
        positive_weights[label] = 16
        negative_weights[label] = 1



    path=(os.getcwd()+'/BPDP_Classifier') # output path
    xt,z =os.path.split(file)

    # writer = pd.ExcelWriter(path+'/'+z+'_BPDP_CIBE_classification_sd_early.xlsx', engine="xlsxwriter")

    x_train_idx, x_test_idx, y_train_idx, y_test_idx = train_test_split(range(len(log)), range(len(log)), test_size=split, random_state=0)

    for d in dev:
        metrics[str('NoDev'+d)]=0
        metrics[d]['LenTrain_beforeUS_1']=sum(dev_df[d][i] for i in x_train_idx)
        metrics[d]['LenTrain_beforeUS_0']=len(x_train_idx)-sum(dev_df[d][i] for i in x_train_idx)

    dev_position = pd.DataFrame(index=x_test_idx, columns=dev, data=0)
    for d in dev:
        for idx in x_test_idx:
            for i in range(1, event_count[idx]+1):
                if y_cum_test[i][d][idx]==1: dev_position[d][idx]=i+1
    dev_position_pred = pd.DataFrame(index=x_test_idx, columns=dev, data=0)
    earliness={}
    sd_earliness={}

    dev_distribution = pd.DataFrame(data=0, index=['Training','Test'], columns=dev)
    for d in dev:
        dev_distribution[d]['Training']=sum(dev_df[d][i] for i in x_train_idx)
        dev_distribution[d]['Test']=sum(dev_df[d][i] for i in x_test_idx)

    # dev_distribution.to_excel(writer, sheet_name=('Distribution'))

    dev_trained=[]
    for d in dev:
        if dev_distribution[d]['Training'] ==0:
            metrics[d]='No Deviation in Training Set'
            continue
        elif dev_distribution[d]['Test'] ==0:
            metrics[d]='No Deviation in Test Set'
            continue
        else:
            dev_trained.append(d)


        Y_cum_dev={}
        for prefix in range(1,max_ev+1):
            Y_cum_dev[prefix]=pd.DataFrame(y_cum_test[prefix][d])
            Y_cum_dev[prefix]['NoDev']=0
            for i in Y_cum_dev[prefix].index.values.tolist():
                Y_cum_dev[prefix]['NoDev'][i]=1-y_cum_test[prefix][d][i]
            if prefix==1:
                print(Y_cum_dev[prefix].columns)

        if u_sample:
            imb_ref_enc_dat=ref_enc_dat.copy()
            imb_ref_enc_dat['ind']=0
            for i in range(len(imb_ref_enc_dat)):
                imb_ref_enc_dat['ind'][i]=i
            imb_traces=pd.DataFrame(data=0, columns=['Dev'], index = range(len(log)))
            for trace in range(len(log)):
                if dev_df[d][trace]>0:
                    imb_traces['Dev'][trace]=1


            imb_traces=imb_traces.drop(x_test_idx)
            imb_ref_enc_dat=imb_ref_enc_dat.drop(x_test_idx)
            imp = SimpleImputer(missing_values=np.nan, strategy='constant',fill_value=0)
            imb_ref_enc_dat=pd.DataFrame(data=imp.fit_transform(imb_ref_enc_dat),columns=imb_ref_enc_dat.columns)

            oss=OneSidedSelection(random_state=0,n_seeds_S=250,n_neighbors=7)

            X_resampled, y_resampled = oss.fit_resample(imb_ref_enc_dat, imb_traces)

            x_train_idx= list(X_resampled['ind'])
            y_train_idx= list(X_resampled['ind'])
        else:
            x_train_idx, x_test_idx, y_train_idx, y_test_idx = train_test_split(range(len(log)), range(len(log)), test_size=split, random_state=0)

        print('index length ', len(x_train_idx),len(x_test_idx),len(y_train_idx),len(y_test_idx))
        metrics[d]['LenTrain']=len(x_train_idx)
        metrics[d]['LenTrain_afterUS_1']=imb_traces['Dev'].sum()
        metrics[d]['LenTrain_afterUS_0']=len(x_train_idx)-imb_traces['Dev'].sum()
        # validation set for early stopping
        x_train_idx, x_val_idx, y_train_idx, y_val_idx = train_test_split(x_train_idx, x_train_idx, test_size=0.2, random_state=0)


        enumerated_trace_idx={}
        for prefix in range(1,max_ev+1):
            drop_idx=[]
            for trace_idx in range(len(log)):
                if event_count[trace_idx]< prefix:
                    drop_idx.append(trace_idx) # drop all trace encoding that do not go until prefix length

            x_te=X_cum[prefix].loc[[j for j in list(set(y_test_idx)-set(drop_idx))]].to_numpy().astype(float)
            x_tr=X_cum[prefix].loc[[j for j in list(set(y_train_idx)-set(drop_idx))]].to_numpy().astype(float)
            x_va=X_cum[prefix].loc[[j for j in list(set(y_val_idx)-set(drop_idx))]].to_numpy().astype(float)
            y_te=Y_cum_dev[prefix].loc[[j for j in list(set(y_test_idx)-set(drop_idx))]].to_numpy().astype(float)
            y_tr=Y_cum_dev[prefix].loc[[j for j in list(set(y_train_idx)-set(drop_idx))]].to_numpy().astype(float)
            y_va=Y_cum_dev[prefix].loc[[j for j in list(set(y_val_idx)-set(drop_idx))]].to_numpy().astype(float)
            enumerated_trace_idx[prefix]=list(set(y_test_idx)-set(drop_idx))
            print('subset length ',prefix, len(x_te),len(x_tr),len(y_te),len(y_tr))

            if prefix ==1:
                X_train = x_tr
                X_test = x_te
                y_train = y_tr
                y_test = y_te
                X_val = x_va
                y_val = y_va
            else:
                X_train = np.append( X_train, x_tr, axis=0)
                X_test = np.append( X_test, x_te, axis=0)
                y_train = np.append( y_train, y_tr, axis=0)
                y_test = np.append( y_test, y_te, axis=0)
                y_val = np.append( y_val, y_va, axis=0)
                X_val = np.append( X_val, x_va, axis=0)# combine all X data from all prefixes into one array
        print(d, len(X_train),len(y_train),len(X_val),len(y_val),len(X_test),len(y_test))


        print('split done')
        scaler = StandardScaler()
        X_test = scaler.fit_transform(X_test)
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.fit_transform(X_val)

        
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        
        model = BinaryClassificationIndiv(no_columns=len(ref_enc_dat.loc[0]))
        model.to(device)

        # weights = torch.FloatTensor(list([positive_weights[d], negative_weights[d]]))
        # criterion = nn.CrossEntropyLoss(weight=weights)
        # FIX: move class weights to the same device as the model (cuda if available)
        weights = torch.tensor([positive_weights[d], negative_weights[d]], dtype=torch.float32, device=device)
        criterion = nn.CrossEntropyLoss(weight=weights)

        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

        if early_stop:
            EPOCHS=300
            model.train()
            train_data = TrainData(torch.FloatTensor(X_train),
                                torch.FloatTensor(y_train))

            train_loader = DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=False)

            # X_val = torch.FloatTensor(X_val)
            X_val = torch.tensor(X_val, dtype=torch.float32, device=device)

            # FIX: also prepare y_val on device, and convert one-hot (Dev/NoDev) to class indices
            y_val_t = torch.tensor(y_val, dtype=torch.float32, device=device)
            y_val_idx = torch.argmax(y_val_t, dim=1).long()

            es = EarlyStopping()
            done = False

            epoch = 0
            while epoch<EPOCHS and not done:
                epoch += 1
                steps = list(enumerate(train_loader))
                pbar = tqdm.tqdm(steps)
                model.train()
                epoch_acc = 0
                epoch_loss=0
                for i, (x_batch, y_batch) in pbar:
                    optimizer.zero_grad()
                    y_batch_pred = model(x_batch.to(device))

                    # loss = criterion(y_batch_pred, y_batch.to(device))
                    # FIX: CrossEntropyLoss expects target as class indices (Long), not one-hot float vector
                    y_batch_t = y_batch.to(device)
                    y_batch_idx = torch.argmax(y_batch_t, dim=1).long()
                    loss = criterion(y_batch_pred, y_batch_idx)

                    # acc = binary_acc(y_batch_pred, y_batch.to(device))/len(y_batch_pred[0])
                    # FIX: compute accuracy from class prediction vs class index
                    acc = (torch.argmax(y_batch_pred, dim=1) == y_batch_idx).float().mean()

                    loss.backward()
                    optimizer.step()
                    epoch_acc += acc.item()

                    loss, current = loss.item(), (i + 1)* len(x_batch)
                    epoch_loss+=loss
                    if i == len(steps)-1:
                        model.eval()
                        pred = model(X_val)
                        
                        # vloss = criterion(pred, torch.FloatTensor(y_val))
                        # FIX: validation loss with y_val indices on same device
                        vloss = criterion(pred, y_val_idx)
                        
                        if es(model,vloss): done = True
                        pbar.set_description(f"Epoch: {epoch}, tloss: {epoch_loss/len(train_loader)}, Acc: {epoch_acc/len(train_loader):.3f}, vloss: {vloss:>7f}, EStop:[{es.status}]")
                    else:
                        pbar.set_description(f"Epoch: {epoch}, tloss {epoch_loss/len(train_loader):}, Acc: {epoch_acc/len(train_loader):.3f}")

        model.eval()
        test_data = TestData(torch.FloatTensor(X_test))
        test_loader = DataLoader(dataset=test_data, batch_size=1)

        y_pred_list = []

        with torch.no_grad():
            for X_batch in test_loader:
                X_batch = X_batch.to(device)

                # y_test_pred = torch.nn.functional.softmax(model(X_batch))
                # y_pred_tag = torch.round(y_test_pred)
                # y_pred_list.append(y_pred_tag.cpu().numpy())
                # FIX: softmax needs dim; and rounding probabilities is wrong for 2-class one-hot;
                #      instead take argmax and convert to one-hot to match y_test shape (Dev/NoDev).
                logits = model(X_batch)
                probs = torch.nn.functional.softmax(logits, dim=1)
                cls = torch.argmax(probs, dim=1, keepdim=True)
                y_pred_tag = torch.zeros_like(probs)
                y_pred_tag.scatter_(1, cls, 1.0)
                y_pred_list.append(y_pred_tag.cpu().numpy())

        y_pred_list = [a.squeeze().tolist() for a in y_pred_list]

        CM = sklearn.metrics.multilabel_confusion_matrix(y_test, y_pred_list)

        metrics[d]['Precision']=CM[0][1][1]/(CM[0][1][1]+CM[0][0][1])
        metrics[d]['Recall']=CM[0][1][1]/(CM[0][1][1]+CM[0][1][0])
        metrics[d]['Support']=CM[0][1][1]+CM[0][1][0]
        try:
            metrics[d]['ROC_AUC'] =  sklearn.metrics.roc_auc_score(y_test, y_pred_list, average='weighted')
        except Exception as er:
            metrics[d]['ROC_AUC'] = er
        metrics[str('NoDev'+d)]['Precision']=CM[1][1][1]/(CM[1][1][1]+CM[1][0][1])
        metrics[str('NoDev'+d)]['Recall']=CM[1][1][1]/(CM[1][1][1]+CM[1][1][0])
        metrics[str('NoDev'+d)]['Support']=CM[1][1][1]+CM[1][1][0]
        print(CM)

        to_be_checked_idx={}
        for idx in x_test_idx:
            cum_idx=0
            for prefix in range(1,event_count[idx]+1):
                if prefix==1:
                    to_be_checked_idx[idx]=[enumerated_trace_idx[1].index(idx)]
                else:
                    to_be_checked_idx[idx].append(enumerated_trace_idx[prefix].index(idx)+cum_idx)
                cum_idx+=len(enumerated_trace_idx[prefix])


        for prefix in range(1, max_ev+1):
            for idx in to_be_checked_idx.keys():
                if event_count[idx]>= prefix:
                    if prefix==1:
                        dev_position_pred[d][idx]=y_pred_list[to_be_checked_idx[idx][prefix-1]][0]
                    else:
                        if y_pred_list[to_be_checked_idx[idx][prefix-1]][0]==1 and y_pred_list[to_be_checked_idx[idx][prefix-2]][0]==0:
                            if dev_position[d][idx]<= prefix:
                                dev_position_pred[d][idx]=dev_position[d][idx]
                            else:
                                dev_position_pred[d][idx]=prefix



        earliness[d]=0
        tobe_devs=0
        sd_earliness[d]=0
        earliness_list=[]
        for idx in to_be_checked_idx.keys():
            if dev_position[d][idx]==0 or dev_position_pred[d][idx]==0:
                continue
            tobe_devs+=1
            earliness[d]+=dev_position_pred[d][idx]/dev_position[d][idx]
            earliness_list.append(dev_position_pred[d][idx]/dev_position[d][idx])
        if not tobe_devs==0:
            earliness[d]=earliness[d]/tobe_devs
            try:
                sd_earliness[d]=stdev(earliness_list)
            except:
                sd_earliness[d]=earliness_list[0]
                print(earliness_list)


        if explained:
            import shap
            import matplotlib.pyplot as plt
            np.random.seed(42)
            e = shap.DeepExplainer(model, torch.FloatTensor(X_train[np.random.choice(X_train.shape[0], 1000, replace=False)]))

            shap_idx=[]
            for j in range(len(y_pred_list)):
                if y_pred_list[j][0]==y_test[j][0]==1:
                    shap_idx.append(j)
            shap_values = e.shap_values(torch.FloatTensor(X_test[shap_idx]))
            fig=shap.summary_plot(shap_values[0], X_test[shap_idx], plot_type = 'dot', feature_names = enc_dat.columns, max_display=10, plot_size=(10,5), show=False)
            plt.savefig(path+'/ShapValues/Dev_'+z+'_'+d+'.png')
            plt.close()

            fig=shap.summary_plot(shap_values[1], X_test[shap_idx], plot_type = 'dot', feature_names = enc_dat.columns, max_display=10, plot_size=(10,5), show=False)
            plt.savefig(path+'/ShapValues/NoDev_'+z+'_'+d+'.png')
            plt.close()


        print(metrics)

    avg_dev_pos={}
    for d in dev:
        if dev_distribution[d]['Test'] ==0:
            metrics[d]='No Deviation in Test Set'
            continue
        if dev_distribution[d]['Training'] ==0:
            metrics[d]='No Deviation in Training Set'
            continue
        devs=0
        positions=0
        for idx in to_be_checked_idx.keys():
            if dev_position[d][idx]>0:
                devs+=1
                positions+=dev_position[d][idx]
        if devs ==0:
            continue
        avg_dev_pos[d]=positions/devs

    # metrics.to_excel(writer, sheet_name=('Metrics'))
    # df=pd.DataFrame(data=earliness, index=[0])
    # df.to_excel(writer, sheet_name=('Earliness'))
    # df=pd.DataFrame(data=sd_earliness, index=[0])
    # df.to_excel(writer, sheet_name=('Earliness_SD'))
    # df=pd.DataFrame(data=avg_dev_pos, index=[0])
    # df.to_excel(writer, sheet_name=('Position'))

    # writer.close()

    # ------------------------------------------------------------------
    # NEW: print unweighted macro avg precision/recall over Dev + NoDev
    # ------------------------------------------------------------------
    # We compute for each deviation type d the per-class precision/recall from
    # the confusion matrices already stored in `metrics`, then average across `dev`.
    try:
        dev_prec, dev_rec = [], []
        nodev_prec, nodev_rec = [], []
        for d in dev:
            # skip non-numeric rows (e.g., "No Deviation in Training/Test Set")
            try:
                p_dev = float(metrics[d]['Precision'])
                r_dev = float(metrics[d]['Recall'])
                p_nd = float(metrics[str('NoDev'+d)]['Precision'])
                r_nd = float(metrics[str('NoDev'+d)]['Recall'])
            except Exception:
                continue
            if np.isfinite(p_dev) and np.isfinite(r_dev): dev_prec.append(p_dev); dev_rec.append(r_dev)
            if np.isfinite(p_nd) and np.isfinite(r_nd): nodev_prec.append(p_nd); nodev_rec.append(r_nd)

        macro_p_dev = float(np.mean(dev_prec)) if len(dev_prec) else float("nan")
        macro_r_dev = float(np.mean(dev_rec)) if len(dev_rec) else float("nan")
        macro_p_nd  = float(np.mean(nodev_prec)) if len(nodev_prec) else float("nan")
        macro_r_nd  = float(np.mean(nodev_rec)) if len(nodev_rec) else float("nan")

        print("\n=== Unweighted macro averages across deviation types ===")
        print(f"Dev   : Precision={macro_p_dev:.4f} | Recall={macro_r_dev:.4f}")
        print(f"NoDev : Precision={macro_p_nd:.4f} | Recall={macro_r_nd:.4f}\n")
    except Exception as er:
        print("Macro-average print failed:", er)

In [ ]:
# seperate FFN
IDP_separate_CIBE(log, ref_log, aligned_traces, split=1/3, u_sample=True, early_stop=True,explained=False)

In [ ]:
def IDP_separate_LSTM_CIBE(log, ref_log, aligned_traces, split=1/3, u_sample=True, early_stop=True,relevance_ths = .5):
    
    xt, z = os.path.split(file)
    import warnings
    warnings.simplefilter('ignore')
    #### get information whether deviation happened after prefix length in DF
    i = 0
    dev = []  # stores all deviations that happened
    for trace in log:
        no_moves = len(aligned_traces[i]['alignment'])
        for j in range(0, len(aligned_traces[i]['alignment'])):
            if aligned_traces[i]['alignment'][j][1] == None or aligned_traces[i]['alignment'][j][0] == \
                    aligned_traces[i]['alignment'][j][1]:
                next  # we do not care for simultaneous or silent moves
            else:
                if not str(aligned_traces[i]['alignment'][j]) in dev:
                    dev.append(str(aligned_traces[i]['alignment'][j]))
        i += 1

    y_cum_test = {}  # dict that stores label for each prefix and deviation combinations; keys are prefix length, entries are Data Frames with index = trace and columns = deviation
    dev_df = pd.DataFrame(data=0, columns=dev, index=range(
        len(log)))  # Data Frame that stores the information whether a deviation happened for each trace on trace level
    event_order = {}  # dict with event sequences for each trace
    event_count = {}  # dict with trace length for each trace
    max_ev = 0  # will be maximum trace length
    k = 0
    for trace in log:
        event_order[k] = []
        i = 0
        for event in trace:
            i += 1
            event_order[k].append(event['concept:name'])
        if i > max_ev:
            max_ev = i
        event_count[k] = len(event_order[k])
        k += 1
    i = 0
    for trace in log:
        no_moves = len(aligned_traces[i]['alignment'])
        for j in range(0, len(aligned_traces[i]['alignment'])):
            if aligned_traces[i]['alignment'][j][1] == None or aligned_traces[i]['alignment'][j][0] == \
                    aligned_traces[i]['alignment'][j][1]:
                next
            else:
                dev_df[str(aligned_traces[i]['alignment'][j])][i] = 1
        i += 1
    for ev in range(1, max_ev + 1):
        y_cum_test[
            ev] = dev_df.copy()  # initialize each prefix length with all traces and information whether deviation happened
    for ev in range(1, max_ev + 1):
        drop_idx = []
        for trace_idx in range(len(log)):
            if event_count[trace_idx] < ev:
                drop_idx.append(trace_idx)  # drop all trace labels that do not go until prefix length
        y_cum_test[ev] = y_cum_test[ev].drop(drop_idx)
    i = 0
    for trace in log:
        no_moves = len(aligned_traces[i]['alignment'])
        j = no_moves - 1  # iterator over moves in alignment, starting at the end
        m = len(event_order[i])  # iterator over event sequence, starting at the end
        while j >= 0:
            if aligned_traces[i]['alignment'][j][
                1] == None:  # if silent move, just go one move further to the beginning in the alignment
                j -= 1
            elif aligned_traces[i]['alignment'][j][0] == aligned_traces[i]['alignment'][j][1]:
                if event_order[i][m - 1] == aligned_traces[i]['alignment'][j][
                    0]:  # if synchronous move, just go one move further to the beginning in the alignment and one event forther to the beginning in the event sequence
                    j -= 1
                    m -= 1
            elif event_order[i][m - 1] == aligned_traces[i]['alignment'][j][0]:  # log move detected
                for q in range(m, max_ev + 1):
                    y_cum_test[q][str(aligned_traces[i]['alignment'][j])][
                        i] = 0  # set all prefixes from the current m to the maximum prefix length in this trace to 0 because deviation happened here but not afterwards
                j -= 1
                m -= 1
            elif m == max_ev:
                j -= 1
            else:  # model move deteceted
                for q in range(m + 1, max_ev + 1):
                    y_cum_test[q][str(aligned_traces[i]['alignment'][j])][
                        i] = 0  # set all prefixes after the current m to the maximum prefix length in this trace to 0 because deviation happened between m and m+1 but not afterwards
                j -= 1
        i += 1
    ### y_cum_test holds information whether deviation happened after prefix length
    x_train_idx, x_test_idx, y_train_idx, y_test_idx = train_test_split(range(len(log)), range(len(log)), test_size=split,
                                                                        random_state=0)
    trainin_dev_df = dev_df.loc[x_train_idx]
    trainin_dev_df.corr()
    corrMatrix = trainin_dev_df.corr()

    corrMatrix.loc[:, :] = np.tril(corrMatrix, k=-1)  # borrowed from Karl D's answer

    already_in = set()
    max_combs_l = []
    for col in corrMatrix:
        perfect_corr = corrMatrix[col][corrMatrix[col] >= relevance_ths].index.tolist()
        if perfect_corr and col not in already_in:
            already_in.update(set(perfect_corr))
            perfect_corr.append(col)
            max_combs_l.append(perfect_corr)
    test_counts = {}
    for comb in max_combs_l:
        for y in range(len(comb)):
            test_counts[comb[y]] = dev_df.loc[x_test_idx].sum()[comb[y]]
        if any(dev_df.loc[x_test_idx].sum()[comb[y]] == 0 for y in range(len(comb))):
            max_combs_l.remove(comb)
            print(comb)

    max_combs = {}
    for comb in max_combs_l:
        max_combs[str(comb)] = comb

    y_cum_test_combs = {}
    for prefix in range(1, max_ev + 1):
        y_cum_test_combs[prefix] = y_cum_test[prefix].copy(deep=True)
        for comb in max_combs.keys():
            y_cum_test_combs[prefix][comb] = 0
            for i in list(y_cum_test_combs[prefix].index):
                if event_count[i] < prefix:
                    continue
                if all(y_cum_test_combs[prefix][j][i] == 1 for j in max_combs[comb]):
                    for j in max_combs[comb]:
                        y_cum_test_combs[prefix][j][i] = 0
                    y_cum_test_combs[prefix][comb][i] = 1
    trainin_dev_df.sum()

    y_cum_test_o_combs = {}
    for prefix in range(1, max_ev + 1):
        y_cum_test_o_combs[prefix] = y_cum_test_combs[prefix][list(max_combs.keys())]


    ## ref_log will have all attributes that will be the columns for X_test and X_train
    ref_log = complex_index_encoding(ref_log,
                                     4000)  # prepare a log with the maximum length of the feature vector from CIBE to know to pad other feature vectors
    ref_dataframe1 = pm4py.convert_to_dataframe(ref_log)
    ref_dataframe = ref_dataframe1.drop_duplicates(subset=['case:concept:name'])
    ref_dataframe = ref_dataframe.filter(like='case:', axis=1)
    ref_dataframe = ref_dataframe.drop('case:concept:name', axis=1)
    ref_dataframe.columns = ref_dataframe.columns.str.replace('case:', '')
    ref_dataframe = ref_dataframe.reset_index()
    ref_raw_dat = ref_dataframe.drop('index', axis=1)
    ## dataset-specific preparation (i.e., redundant attributes, convertion to numeric)
    if z == 'aligned_traces_12A.pkl' or z == 'aligned_traces_12O.pkl' or z == 'aligned_traces_12AO.pkl':
        ref_raw_dat['AMOUNT_REQ'] = pd.to_numeric(ref_raw_dat['AMOUNT_REQ'])
        ref_clean_dat = ref_raw_dat.drop('REG_DATE', axis=1)
    elif z == 'aligned_traces_20int.pkl':
        ref_clean_dat = ref_raw_dat.drop(
            ['Permit travel permit number', 'DeclarationNumber', 'travel permit number', 'id', 'Permit ID', 'Permit id'],
            axis=1)
    elif z == 'aligned_traces_20dom.pkl':
        ref_clean_dat = ref_raw_dat.drop(['DeclarationNumber', 'id'], axis=1)
    elif z == 'aligned_traces_20prep.pkl':
        ref_clean_dat = ref_raw_dat.drop(['RfpNumber', 'Rfp_id', 'Permit travel permit number', 'Permit id'], axis=1)
    elif z == 'aligned_traces_20RfP.pkl':
        ref_clean_dat = ref_raw_dat.drop(['RfpNumber', 'Rfp_id'], axis=1)
    else:
        ref_clean_dat = ref_raw_dat.copy()

    ref_enc_dat = ref_clean_dat.copy()

    BATCH_SIZE = 128
    LEARNING_RATE = 0.00001

    X_cum = {}
    metrics = pd.DataFrame(data=0, columns=dev, index=['Precision', 'Recall', 'Support', 'ROC_AUC','Time'])

    # prepare X for all prefix lengths
    for prefix in range(1, max_ev + 1):
        complex_index_encoding(log, prefix)
        dataframe1 = pm4py.convert_to_dataframe(log)
        dataframe = dataframe1.drop_duplicates(subset=['case:concept:name'])
        dataframe = dataframe.filter(like='case:', axis=1)
        dataframe = dataframe.drop('case:concept:name', axis=1)
        dataframe.columns = dataframe.columns.str.replace('case:', '')
        dataframe = dataframe.reset_index()
        raw_dat = dataframe.drop('index', axis=1)
        if z == 'aligned_traces_12A.pkl' or z == 'aligned_traces_12O.pkl' or z == 'aligned_traces_12AO.pkl':
            raw_dat['AMOUNT_REQ'] = pd.to_numeric(raw_dat['AMOUNT_REQ'])
            clean_dat = raw_dat.drop('REG_DATE', axis=1)
        elif z == 'aligned_traces_20int.pkl':
            clean_dat = raw_dat.drop(
                ['Permit travel permit number', 'DeclarationNumber', 'travel permit number', 'id', 'Permit ID',
                 'Permit id'], axis=1)
        elif z == 'aligned_traces_20dom.pkl':
            clean_dat = raw_dat.drop(['DeclarationNumber', 'id'], axis=1)
        elif z == 'aligned_traces_20prep.pkl':
            clean_dat = raw_dat.drop(['RfpNumber', 'Rfp_id', 'Permit travel permit number', 'Permit id'], axis=1)
        elif z == 'aligned_traces_20RfP.pkl':
            clean_dat = raw_dat.drop(['RfpNumber', 'Rfp_id'], axis=1)
        else:
            clean_dat = raw_dat.copy()
        enc_dat = clean_dat.copy()
        for key in ref_enc_dat.columns:
            if not key in enc_dat.columns:
                enc_dat[key] = 'No'  # pad all prefixes to maximum lengths with 0
        imp = SimpleImputer(missing_values=np.nan, strategy='constant', fill_value='No')
        enc_dat = pd.DataFrame(data=imp.fit_transform(enc_dat), columns=enc_dat.columns)

        X_cum[prefix] = enc_dat.copy()
        drop_idx = []
        for trace_idx in range(len(log)):
            if event_count[trace_idx] < prefix:
                drop_idx.append(trace_idx)

        X_cum[prefix] = X_cum[prefix].drop(drop_idx)

    for d in dev:
        metrics[str('NoDev' + d)] = 0

    path = (os.getcwd() + '/BPDP_LSTM')  # output path
    xt, z = os.path.split(file)

    #writer = pd.ExcelWriter('BPDP_LSTM' + '/' + z + '_BPDP_LSTM_time_stopped.xlsx', engine="xlsxwriter")

    x_train_idx, x_test_idx, y_train_idx, y_test_idx = train_test_split(range(len(log)), range(len(log)), test_size=split,
                                                                        random_state=0)

    x_train_idx_c, x_test_idx_c, y_train_idx_c, y_test_idx_c = train_test_split(range(len(log)), range(len(log)),
                                                                                test_size=split,
                                                                                random_state=0)
    x_train_idx_c, x_val_idx_c, y_train_idx_c, y_val_idx_c = train_test_split(x_train_idx_c, x_train_idx_c, test_size=0.2,
                                                                              random_state=0)

    dev_position = pd.DataFrame(index=x_test_idx, columns=dev, data=0)
    for d in dev:
        for idx in x_test_idx:
            for i in range(1, event_count[idx] + 1):
                if y_cum_test[i][d][idx] == 1: dev_position[d][idx] = i + 1
    dev_position_pred = pd.DataFrame(index=x_test_idx, columns=dev, data=0)
    earliness = {}

    dev_distribution = pd.DataFrame(data=0, index=['Training', 'Test'], columns=dev)
    for d in dev:
        dev_distribution[d]['Training'] = sum(dev_df[d][i] for i in x_train_idx)
        dev_distribution[d]['Test'] = sum(dev_df[d][i] for i in x_test_idx)

    #dev_distribution.to_excel(writer, sheet_name=('Distribution'))


    def flatten_comprehension(matrix):
        return [item for row in matrix for item in row]


    ref_enc_dat

    evs_c = []
    resource_c = []
    month_c = []
    trace_attr = []
    for ca in X_cum[1].columns:
        if ca.startswith('event'): evs_c.append(ca)
    for ca in X_cum[1].columns:
        if ca.startswith('resource'): resource_c.append(ca)
    for ca in X_cum[1].columns:
        if ca.startswith('month'): month_c.append(ca)
    for ca in X_cum[1].columns:
        if not (ca in evs_c or ca in resource_c or ca in month_c): trace_attr.append(ca)
    print(evs_c)
    print(resource_c)
    print(month_c)
    print(trace_attr)

    X_events = {}
    X_resource = {}
    X_month = {}
    X_tracea = {}
    for prefix in range(1, max_ev + 1):
        X_events[prefix] = X_cum[prefix][evs_c]
        X_resource[prefix] = X_cum[prefix][resource_c]
        X_month[prefix] = X_cum[prefix][month_c]
        X_tracea[prefix] = X_cum[prefix][trace_attr]

    cat_tas = []
    for cat in X_tracea[1].columns:
        if type(X_tracea[1][cat][0]) == str:
            cat_tas.append(cat)

    uniques_cats = {}
    for cat in cat_tas:
        uniques_cats[cat] = []
    for prefix in range(1, max_ev + 1):
        for cat in cat_tas:
            for reals in list(X_tracea[prefix][cat].unique()):
                if not reals in uniques_cats[cat]:
                    uniques_cats[cat].append(reals)

    for cat in cat_tas:
        for prefix in range(1, max_ev + 1):
            for j in list(X_tracea[prefix].index):
                X_tracea[prefix][cat][j] = uniques_cats[cat].index(X_tracea[prefix][cat][j])

    positive_weights = {}
    negative_weights = {}
    for label in dev:
        positive_weights[label] = 16
        negative_weights[label] = 1
    models_collect = {}
    dev_trained = []
    outputs_train = pd.DataFrame()
    outputs_test = pd.DataFrame()
    outputs_val = pd.DataFrame()

    for d in dev:
        time_start = time.clock()
        if dev_distribution[d]['Training'] == 0:
            metrics[d] = 'No Deviation in Training Set'
            continue
        elif dev_distribution[d]['Test'] == 0:
            metrics[d] = 'No Deviation in Test Set'
            continue

        Y_cum_dev = {}
        for prefix in range(1, max_ev + 1):
            Y_cum_dev[prefix] = pd.DataFrame(y_cum_test[prefix][d])
            Y_cum_dev[prefix]['NoDev'] = 0
            for i in Y_cum_dev[prefix].index.values.tolist():
                Y_cum_dev[prefix]['NoDev'][i] = 1 - y_cum_test[prefix][d][i]
            if prefix == 1:
                print(Y_cum_dev[prefix].columns)

        if u_sample:
            imb_ref_enc_dat = ref_enc_dat.copy()
            imb_ref_enc_dat = pd.get_dummies(imb_ref_enc_dat)
            imb_ref_enc_dat['ind'] = 0
            for i in range(len(imb_ref_enc_dat)):
                imb_ref_enc_dat['ind'][i] = i
            imb_traces = pd.DataFrame(data=0, columns=['Dev'], index=range(len(log)))
            for trace in range(len(log)):
                if dev_df[d][trace] > 0:
                    imb_traces['Dev'][trace] = 1

            imb_traces = imb_traces.drop(x_test_idx)
            imb_ref_enc_dat = imb_ref_enc_dat.drop(x_test_idx)
            imp = SimpleImputer(missing_values=np.nan, strategy='constant', fill_value=0)
            imb_ref_enc_dat = pd.DataFrame(data=imp.fit_transform(imb_ref_enc_dat), columns=imb_ref_enc_dat.columns)

            oss = OneSidedSelection(random_state=0, n_seeds_S=250, n_neighbors=7)

            X_resampled, y_resampled = oss.fit_resample(imb_ref_enc_dat, imb_traces)

            x_train_idx = list(X_resampled['ind'])
            y_train_idx = list(X_resampled['ind'])

        print('index length ', len(x_train_idx), len(x_test_idx), len(y_train_idx), len(y_test_idx))

        # validation set for early stopping
        x_train_idx, x_val_idx, y_train_idx, y_val_idx = train_test_split(x_train_idx, x_train_idx, test_size=0.2,
                                                                          random_state=0)

        enumerated_trace_idx = {}
        cum_trace_idxs = []
        pref_list = []
        pref_list_train_c = []
        pref_list_test_c = []
        pref_list_val_c = []
        for prefix in range(1, max_ev + 1):
            drop_idx = []
            for trace_idx in range(len(log)):
                if event_count[trace_idx] < prefix:
                    drop_idx.append(trace_idx)  # drop all trace encoding that do not go until prefix length

            x_te = X_events[prefix].loc[[j for j in list(set(y_test_idx) - set(drop_idx))]].to_numpy()
            x_tr = X_events[prefix].loc[[j for j in list(set(y_train_idx) - set(drop_idx))]].to_numpy()
            x_va = X_events[prefix].loc[[j for j in list(set(y_val_idx) - set(drop_idx))]].to_numpy()
            y_te = Y_cum_dev[prefix].loc[[j for j in list(set(y_test_idx) - set(drop_idx))]].to_numpy().astype(float)
            y_tr = Y_cum_dev[prefix].loc[[j for j in list(set(y_train_idx) - set(drop_idx))]].to_numpy().astype(float)
            y_va = Y_cum_dev[prefix].loc[[j for j in list(set(y_val_idx) - set(drop_idx))]].to_numpy().astype(float)
            x_te_c = X_events[prefix].loc[[j for j in list(set(y_test_idx_c) - set(drop_idx))]].to_numpy()
            x_tr_c = X_events[prefix].loc[[j for j in list(set(y_train_idx_c) - set(drop_idx))]].to_numpy()
            x_va_c = X_events[prefix].loc[[j for j in list(set(y_val_idx_c) - set(drop_idx))]].to_numpy()
            y_te_c = y_cum_test_o_combs[prefix].loc[[j for j in list(set(y_test_idx) - set(drop_idx))]].to_numpy().astype(
                float)
            y_tr_c = y_cum_test_o_combs[prefix].loc[
                [j for j in list(set(y_train_idx_c) - set(drop_idx))]].to_numpy().astype(float)
            y_va_c = y_cum_test_o_combs[prefix].loc[[j for j in list(set(y_val_idx_c) - set(drop_idx))]].to_numpy().astype(
                float)
            cum_trace_idxs.append(list(set(y_test_idx) - set(drop_idx)))
            pref_list.append([prefix] * len(list(set(y_test_idx) - set(drop_idx))))
            enumerated_trace_idx[prefix] = list(set(y_test_idx) - set(drop_idx))
            pref_list_train_c.append([prefix] * len(list(set(y_train_idx_c) - set(drop_idx))))
            pref_list_test_c.append([prefix] * len(list(set(y_test_idx) - set(drop_idx))))
            pref_list_val_c.append([prefix] * len(list(set(y_val_idx_c) - set(drop_idx))))
            print('subset length ', prefix, len(x_te), len(x_tr), len(y_te), len(y_tr))

            if prefix == 1:
                X_train_event = x_tr
                X_test_event = x_te
                y_train = y_tr
                y_test = y_te
                X_val_event = x_va
                y_val = y_va
                X_train_event_c = x_tr_c
                X_test_event_c = x_te_c
                X_val_event_c = x_va_c
                y_train_c = y_tr_c
                y_test_c = y_te_c
                y_val_c = y_va_c
            else:
                X_train_event = np.append(X_train_event, x_tr, axis=0)
                X_test_event = np.append(X_test_event, x_te, axis=0)
                y_train = np.append(y_train, y_tr, axis=0)
                y_test = np.append(y_test, y_te, axis=0)
                y_val = np.append(y_val, y_va, axis=0)
                X_val_event = np.append(X_val_event, x_va, axis=0)
                X_train_event_c = np.append(X_train_event_c, x_tr_c, axis=0)
                X_test_event_c = np.append(X_test_event_c, x_te_c, axis=0)
                X_val_event_c = np.append(X_val_event_c, x_va_c, axis=0)
                y_train_c = np.append(y_train_c, y_tr_c, axis=0)
                y_test_c = np.append(y_test_c, y_te_c, axis=0)
                y_val_c = np.append(y_val_c, y_va_c, axis=0)  # combine all X data from all prefixes into one array
        print(d, len(X_train_event), len(y_train), len(y_train_c), len(X_val_event), len(y_val), len(y_val_c),
              len(X_test_event), len(y_test), len(y_test_c))

        for prefix in range(1, max_ev + 1):
            drop_idx = []
            for trace_idx in range(len(log)):
                if event_count[trace_idx] < prefix:
                    drop_idx.append(trace_idx)  # drop all trace encoding that do not go until prefix length

            x_te = X_resource[prefix].loc[[j for j in list(set(y_test_idx) - set(drop_idx))]].to_numpy()
            x_tr = X_resource[prefix].loc[[j for j in list(set(y_train_idx) - set(drop_idx))]].to_numpy()
            x_va = X_resource[prefix].loc[[j for j in list(set(y_val_idx) - set(drop_idx))]].to_numpy()
            x_te_c = X_resource[prefix].loc[[j for j in list(set(y_test_idx_c) - set(drop_idx))]].to_numpy()
            x_tr_c = X_resource[prefix].loc[[j for j in list(set(y_train_idx_c) - set(drop_idx))]].to_numpy()
            x_va_c = X_resource[prefix].loc[[j for j in list(set(y_val_idx_c) - set(drop_idx))]].to_numpy()
            print('subset length ', prefix, len(x_te), len(x_tr), len(y_te), len(y_tr))

            if prefix == 1:
                X_train_resource = x_tr
                X_test_resource = x_te
                X_val_resource = x_va
                X_train_resource_c = x_tr_c
                X_test_resource_c = x_te_c
                X_val_resource_c = x_va_c
            else:
                X_train_resource = np.append(X_train_resource, x_tr, axis=0)
                X_test_resource = np.append(X_test_resource, x_te, axis=0)
                X_val_resource = np.append(X_val_resource, x_va, axis=0)
                X_train_resource_c = np.append(X_train_resource_c, x_tr_c, axis=0)
                X_test_resource_c = np.append(X_test_resource_c, x_te_c, axis=0)
                X_val_resource_c = np.append(X_val_resource_c, x_va_c,
                                             axis=0)  # combine all X data from all prefixes into one array
        print(d, len(X_train_resource), len(y_train), len(X_val_resource), len(y_val), len(X_test_resource), len(y_test))

        for prefix in range(1, max_ev + 1):
            drop_idx = []
            for trace_idx in range(len(log)):
                if event_count[trace_idx] < prefix:
                    drop_idx.append(trace_idx)  # drop all trace encoding that do not go until prefix length

            x_te = X_month[prefix].loc[[j for j in list(set(y_test_idx) - set(drop_idx))]].to_numpy()
            x_tr = X_month[prefix].loc[[j for j in list(set(y_train_idx) - set(drop_idx))]].to_numpy()
            x_va = X_month[prefix].loc[[j for j in list(set(y_val_idx) - set(drop_idx))]].to_numpy()
            x_te_c = X_month[prefix].loc[[j for j in list(set(y_test_idx_c) - set(drop_idx))]].to_numpy()
            x_tr_c = X_month[prefix].loc[[j for j in list(set(y_train_idx_c) - set(drop_idx))]].to_numpy()
            x_va_c = X_month[prefix].loc[[j for j in list(set(y_val_idx_c) - set(drop_idx))]].to_numpy()
            print('subset length ', prefix, len(x_te), len(x_tr), len(y_te), len(y_tr))

            if prefix == 1:
                X_train_m = x_tr
                X_test_m = x_te
                X_val_m = x_va
                X_train_m_c = x_tr_c
                X_test_m_c = x_te_c
                X_val_m_c = x_va_c
            else:
                X_train_m = np.append(X_train_m, x_tr, axis=0)
                X_test_m = np.append(X_test_m, x_te, axis=0)
                X_val_m = np.append(X_val_m, x_va, axis=0)
                X_train_m_c = np.append(X_train_m_c, x_tr_c, axis=0)
                X_test_m_c = np.append(X_test_m_c, x_te_c, axis=0)
                X_val_m_c = np.append(X_val_m_c, x_va_c, axis=0)  # combine all X data from all prefixes into one array
        print(d, len(X_train_m), len(y_train), len(X_val_m), len(y_val), len(X_test_m), len(y_test))

        print('split done')

        for prefix in range(1, max_ev + 1):
            drop_idx = []
            for trace_idx in range(len(log)):
                if event_count[trace_idx] < prefix:
                    drop_idx.append(trace_idx)  # drop all trace encoding that do not go until prefix length

            x_te = X_tracea[prefix].loc[[j for j in list(set(y_test_idx) - set(drop_idx))]].to_numpy()
            x_tr = X_tracea[prefix].loc[[j for j in list(set(y_train_idx) - set(drop_idx))]].to_numpy()
            x_va = X_tracea[prefix].loc[[j for j in list(set(y_val_idx) - set(drop_idx))]].to_numpy()
            x_te_c = X_tracea[prefix].loc[[j for j in list(set(y_test_idx_c) - set(drop_idx))]].to_numpy()
            x_tr_c = X_tracea[prefix].loc[[j for j in list(set(y_train_idx_c) - set(drop_idx))]].to_numpy()
            x_va_c = X_tracea[prefix].loc[[j for j in list(set(y_val_idx_c) - set(drop_idx))]].to_numpy()
            print('subset length ', prefix, len(x_te), len(x_tr), len(y_te), len(y_tr))

            if prefix == 1:
                X_train_TA = x_tr
                X_test_TA = x_te
                X_val_TA = x_va
                X_train_TA_c = x_tr_c
                X_test_TA_c = x_te_c
                X_val_TA_c = x_va_c
            else:
                X_train_TA = np.append(X_train_TA, x_tr, axis=0)
                X_test_TA = np.append(X_test_TA, x_te, axis=0)
                X_val_TA = np.append(X_val_TA, x_va, axis=0)
                X_train_TA_c = np.append(X_train_TA_c, x_tr_c, axis=0)
                X_test_TA_c = np.append(X_test_TA_c, x_te_c, axis=0)
                X_val_TA_c = np.append(X_val_TA_c, x_va_c, axis=0)  # combine all X data from all prefixes into one array
        print(d, len(X_train_TA), len(y_train), len(X_val_TA), len(y_val), len(X_test_TA), len(y_test))

        events_encoder = list(
            np.unique(np.append(np.append(X_train_event_c, X_test_event_c, axis=0), X_val_event_c, axis=0)))
        events_encoder.index('No')
        resource_encoder = list(
            np.unique(np.append(np.append(X_train_resource_c, X_test_resource_c, axis=0), X_val_resource_c, axis=0)))
        resource_encoder.index('No')
        cat_ecnoders = {}
        for cat in cat_tas:
            cat_ecnoders[cat] = list(np.unique(np.append(np.append(X_train_TA_c, X_test_TA_c, axis=0), X_val_TA_c, axis=0)))

        month_encoder = list(np.unique(np.append(np.append(X_train_m_c, X_test_m_c, axis=0), X_val_m_c, axis=0)))
        month_encoder
        for i in range(len(X_test_event)):
            for j in range(len(X_test_event[0])):
                X_test_event[i][j] = events_encoder.index(X_test_event[i][j])
            for j in range(len(X_test_resource[0])):
                X_test_resource[i][j] = resource_encoder.index(X_test_resource[i][j])
            for j in range(len(X_test_m[0])):
                X_test_m[i][j] = month_encoder.index(X_test_m[i][j])
        for i in range(len(X_train_event)):
            for j in range(len(X_train_event[0])):
                X_train_event[i][j] = events_encoder.index(X_train_event[i][j])
            for j in range(len(X_train_resource[0])):
                X_train_resource[i][j] = resource_encoder.index(X_train_resource[i][j])
            for j in range(len(X_train_m[0])):
                X_train_m[i][j] = month_encoder.index(X_train_m[i][j])
        for i in range(len(X_val_event)):
            for j in range(len(X_val_event[0])):
                X_val_event[i][j] = events_encoder.index(X_val_event[i][j])
            for j in range(len(X_val_resource[0])):
                X_val_resource[i][j] = resource_encoder.index(X_val_resource[i][j])
            for j in range(len(X_val_m[0])):
                X_val_m[i][j] = month_encoder.index(X_val_m[i][j])

        # for combs_output
        for i in range(len(X_test_event_c)):
            for j in range(len(X_test_event_c[0])):
                X_test_event_c[i][j] = events_encoder.index(X_test_event_c[i][j])
            for j in range(len(X_test_resource[0])):
                X_test_resource_c[i][j] = resource_encoder.index(X_test_resource_c[i][j])
            for j in range(len(X_test_m[0])):
                X_test_m_c[i][j] = month_encoder.index(X_test_m_c[i][j])
        for i in range(len(X_train_event_c)):
            for j in range(len(X_train_event_c[0])):
                X_train_event_c[i][j] = events_encoder.index(X_train_event_c[i][j])
            for j in range(len(X_train_resource[0])):
                X_train_resource_c[i][j] = resource_encoder.index(X_train_resource_c[i][j])
            for j in range(len(X_train_m[0])):
                X_train_m_c[i][j] = month_encoder.index(X_train_m_c[i][j])
        for i in range(len(X_val_event_c)):
            for j in range(len(X_val_event_c[0])):
                X_val_event_c[i][j] = events_encoder.index(X_val_event_c[i][j])
            for j in range(len(X_val_resource_c[0])):
                X_val_resource_c[i][j] = resource_encoder.index(X_val_resource_c[i][j])
            for j in range(len(X_val_m_c[0])):
                X_val_m_c[i][j] = month_encoder.index(X_val_m_c[i][j])
        scaler = StandardScaler()
        X_test_TA = scaler.fit_transform(X_test_TA)
        X_train_TA = scaler.fit_transform(X_train_TA)
        X_val_TA = scaler.fit_transform(X_val_TA)
        X_test_TA_c = scaler.fit_transform(X_test_TA_c)
        X_train_TA_c = scaler.fit_transform(X_train_TA_c)
        X_val_TA_c = scaler.fit_transform(X_val_TA_c)
        X_test_event = X_test_event.astype(int)
        X_train_event = X_train_event.astype(int)
        X_val_event = X_val_event.astype(int)
        X_test_resource = X_test_resource.astype(int)
        X_train_resource = X_train_resource.astype(int)
        X_val_resource = X_val_resource.astype(int)
        X_test_m = X_test_m.astype(int)
        X_train_m = X_train_m.astype(int)
        X_val_m = X_val_m.astype(int)
        #for combs again
        X_test_event_c = X_test_event_c.astype(int)
        X_train_event_c = X_train_event_c.astype(int)
        X_val_event_c = X_val_event_c.astype(int)
        X_test_resource_c = X_test_resource_c.astype(int)
        X_train_resource_c = X_train_resource_c.astype(int)
        X_val_resource_c = X_val_resource_c.astype(int)
        X_test_m_c = X_test_m_c.astype(int)
        X_train_m_c = X_train_m_c.astype(int)
        X_val_m_c = X_val_m_c.astype(int)

        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        model = BPDP_LSTM(vocab_events=len(events_encoder), vocab_resources=len(resource_encoder), no_TA=len(X_train_TA[0]),
                          vocab_month=len(month_encoder))
        model.to(device)

        # weights = torch.FloatTensor(list([positive_weights[d], negative_weights[d]]))
        # criterion = nn.CrossEntropyLoss(weight=weights)
        # FIX: move class weights to the same device as the model
        weights = torch.tensor([positive_weights[d], negative_weights[d]], dtype=torch.float32, device=device)
        criterion = nn.CrossEntropyLoss(weight=weights)

        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        torch.autograd.set_detect_anomaly(True)
        if early_stop:
            EPOCHS = 300
            model.train()
            train_data_event = TrainData(torch.FloatTensor(X_train_event),
                                         torch.FloatTensor(y_train))
            train_data_resource = TestData(torch.FloatTensor(X_train_resource))
            train_data_TA = TestData(torch.FloatTensor(X_train_TA))
            train_data_m = TestData(torch.FloatTensor(X_train_m))
            train_loader_event = DataLoader(dataset=train_data_event, batch_size=BATCH_SIZE, shuffle=False)
            train_loader_resource = DataLoader(dataset=train_data_resource, batch_size=BATCH_SIZE, shuffle=False)
            train_loader_TA = DataLoader(dataset=train_data_TA, batch_size=BATCH_SIZE, shuffle=False)
            train_loader_m = DataLoader(dataset=train_data_m, batch_size=BATCH_SIZE, shuffle=False)

            # FIX: put validation tensors on device + convert one-hot y_val to class indices
            X_val_event_t = torch.tensor(X_val_event, dtype=torch.int64, device=device)
            X_val_resource_t = torch.tensor(X_val_resource, dtype=torch.int64, device=device)
            X_val_TA_t = torch.tensor(X_val_TA, dtype=torch.float32, device=device)
            X_val_m_t = torch.tensor(X_val_m, dtype=torch.int64, device=device)
            y_val_t = torch.tensor(y_val, dtype=torch.float32, device=device)
            y_val_idx = torch.argmax(y_val_t, dim=1).long()

            es = EarlyStopping()
            done = False

            epoch = 0
            while epoch < EPOCHS and not done:
                epoch += 1
                steps = list(enumerate(train_loader_event))
                pbar = tqdm.tqdm(steps)
                steps_r = list((train_loader_resource))
                steps_ta = list((train_loader_TA))
                steps_m = list((train_loader_m))
                model.train()
                epoch_acc = 0
                epoch_loss = 0
                for i, (x_batch, y_batch) in pbar:
                    optimizer.zero_grad()
                    y_batch_pred = model(x_batch.to(torch.int64).to(device), steps_r[i].to(torch.int64).to(device),
                                         steps_ta[i].to(torch.float).to(device), steps_m[i].to(torch.int64).to(device))

                    # loss = criterion(y_batch_pred, y_batch.to(device))
                    # FIX: CrossEntropyLoss expects class indices (Long), not one-hot vectors
                    y_batch_t = y_batch.to(device)
                    y_batch_idx = torch.argmax(y_batch_t, dim=1).long()
                    loss = criterion(y_batch_pred, y_batch_idx)

                    # acc = binary_acc(y_batch_pred, y_batch.to(device)) / len(y_batch_pred[0])
                    # FIX: accuracy from argmax vs class indices
                    acc = (torch.argmax(y_batch_pred, dim=1) == y_batch_idx).float().mean()

                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                    optimizer.step()
                    epoch_acc += acc.item()

                    loss, current = loss.item(), (i + 1) * len(x_batch)
                    epoch_loss += loss
                    if i == len(steps) - 1:
                        model.eval()
                        pred = model(X_val_event_t, X_val_resource_t, X_val_TA_t, X_val_m_t)

                        # vloss = criterion(pred, torch.FloatTensor(y_val))
                        # FIX: validation loss with y_val indices on same device
                        vloss = criterion(pred, y_val_idx)

                        if es(model, vloss): done = True
                        pbar.set_description(
                            f"Epoch: {epoch}, tloss: {epoch_loss / len(train_loader_event)}, Acc: {epoch_acc / len(train_loader_event):.3f}, vloss: {vloss:>7f}, EStop:[{es.status}]")
                    else:
                        pbar.set_description(
                            f"Epoch: {epoch}, tloss {epoch_loss / len(train_loader_event):}, Acc: {epoch_acc / len(train_loader_event):.3f}")

        y_batch_pred
        model.eval()
        test_data_event = TestData(torch.FloatTensor(X_test_event))
        test_data_resource = TestData(torch.FloatTensor(X_test_resource))
        test_data_TA = TestData(torch.FloatTensor(X_test_TA))
        test_data_m = TestData(torch.FloatTensor(X_test_m))
        test_loader_event = DataLoader(dataset=test_data_event, batch_size=1)
        test_loader_resource = DataLoader(dataset=test_data_resource, batch_size=1)
        test_loader_TA = DataLoader(dataset=test_data_TA, batch_size=1)
        test_loader_m = DataLoader(dataset=test_data_m, batch_size=1)
        iterations_r = iter(test_loader_resource)
        iterations_ta = iter(test_loader_TA)
        iterations_m = iter(test_loader_m)
        y_pred_list = []

        with torch.no_grad():
            for i, X_batch in enumerate(test_loader_event):
                X_batch = X_batch.to(device).to(torch.int64)

                # y_test_pred = torch.nn.functional.softmax(
                #     model(X_batch, next(iterations_r).to(torch.int64), next(iterations_ta).to(torch.float),
                #           next(iterations_m).to(torch.int64)))
                # y_pred_tag = torch.round(y_test_pred)
                # y_pred_list.append(y_pred_tag.cpu().numpy())
                # FIX: softmax needs dim; and rounding is not correct -> argmax then one-hot
                logits = model(
                    X_batch,
                    next(iterations_r).to(torch.int64).to(device),
                    next(iterations_ta).to(torch.float).to(device),
                    next(iterations_m).to(torch.int64).to(device),
                )
                probs = torch.nn.functional.softmax(logits, dim=1)
                cls = torch.argmax(probs, dim=1, keepdim=True)
                y_pred_tag = torch.zeros_like(probs)
                y_pred_tag.scatter_(1, cls, 1.0)
                y_pred_list.append(y_pred_tag.cpu().numpy())

        y_pred_list = [a.squeeze().tolist() for a in y_pred_list]

        CM = sklearn.metrics.multilabel_confusion_matrix(y_test, y_pred_list)

        time_elapsed = time_start - time.clock()
        metrics[d]['Time']=time_elapsed

        metrics[d]['Precision'] = CM[0][1][1] / (CM[0][1][1] + CM[0][0][1])
        metrics[d]['Recall'] = CM[0][1][1] / (CM[0][1][1] + CM[0][1][0])
        metrics[d]['Support'] = CM[0][1][1] + CM[0][1][0]
        try:
            metrics[d]['ROC_AUC'] = sklearn.metrics.roc_auc_score(y_test, y_pred_list, average='weighted')
        except Exception as er:
            metrics[d]['ROC_AUC'] = er
        metrics[str('NoDev' + d)]['Precision'] = CM[1][1][1] / (CM[1][1][1] + CM[1][0][1])
        metrics[str('NoDev' + d)]['Recall'] = CM[1][1][1] / (CM[1][1][1] + CM[1][1][0])
        metrics[str('NoDev' + d)]['Support'] = CM[1][1][1] + CM[1][1][0]
        print(CM)

        print(metrics)
        models_collect[d] = model

        X_test_event_c = X_test_event_c.astype(int)
        X_train_event_c = X_train_event_c.astype(int)
        X_val_event_c = X_val_event_c.astype(int)
        X_test_resource_c = X_test_resource_c.astype(int)
        X_train_resource_c = X_train_resource_c.astype(int)
        X_val_resource_c = X_val_resource_c.astype(int)
        X_test_m_c = X_test_m_c.astype(int)
        X_train_m_c = X_train_m_c.astype(int)
        X_val_m_c = X_val_m_c.astype(int)

        train_data_f_combs_event = TestData(torch.FloatTensor(X_train_event_c))
        train_loader_f_combs_event = DataLoader(dataset=train_data_f_combs_event, batch_size=len(X_train_event_c))
        train_data_f_combs_resource = TestData(torch.FloatTensor(X_train_resource_c))
        train_loader_f_combs_resource = DataLoader(dataset=train_data_f_combs_resource, batch_size=len(X_train_resource_c))
        train_data_f_combs_m = TestData(torch.FloatTensor(X_train_m_c))
        train_loader_f_combs_m = DataLoader(dataset=train_data_f_combs_m, batch_size=len(X_train_m_c))
        train_data_f_combs_TA = TestData(torch.FloatTensor(X_train_TA_c))
        train_loader_f_combs_TA = DataLoader(dataset=train_data_f_combs_TA, batch_size=len(X_train_TA_c))

        y_output_train = []
        with torch.no_grad():
            steps_r = list((train_loader_f_combs_resource))
            steps_ta = list((train_loader_f_combs_TA))
            steps_m = list((train_loader_f_combs_m))
            for i, X_batch in enumerate(train_loader_f_combs_event):
                
                y_test_pred = model(X_batch.to(torch.int64).to(device), steps_r[i].to(torch.int64).to(device),
                                    steps_ta[i].to(torch.float).to(device), steps_m[i].to(torch.int64).to(device))
                
                # change
                y_test_pred = y_test_pred.cpu()
                
                y_output_train.append(y_test_pred.numpy())

        test_data_f_combs_event = TestData(torch.FloatTensor(X_test_event_c))
        test_loader_f_combs_event = DataLoader(dataset=test_data_f_combs_event, batch_size=len(X_test_event_c))
        test_data_f_combs_resource = TestData(torch.FloatTensor(X_test_resource_c))
        test_loader_f_combs_resource = DataLoader(dataset=test_data_f_combs_resource, batch_size=len(X_test_resource_c))
        test_data_f_combs_m = TestData(torch.FloatTensor(X_test_m_c))
        test_loader_f_combs_m = DataLoader(dataset=test_data_f_combs_m, batch_size=len(X_test_m_c))
        test_data_f_combs_TA = TestData(torch.FloatTensor(X_test_TA_c))
        test_loader_f_combs_TA = DataLoader(dataset=test_data_f_combs_TA, batch_size=len(X_test_TA_c))

        y_output_test = []
        with torch.no_grad():
            steps_r = list((test_loader_f_combs_resource))
            steps_ta = list((test_loader_f_combs_TA))
            steps_m = list((test_loader_f_combs_m))
            for i, X_batch in enumerate(test_loader_f_combs_event):
                y_test_pred = model(X_batch.to(torch.int64).to(device), steps_r[i].to(torch.int64).to(device),
                                    steps_ta[i].to(torch.float).to(device), steps_m[i].to(torch.int64).to(device))
                
                # change
                y_test_pred = y_test_pred.cpu()
                
                y_output_test.append(y_test_pred.numpy())

        val_data_f_combs_event = TestData(torch.FloatTensor(X_val_event_c))
        val_loader_f_combs_event = DataLoader(dataset=val_data_f_combs_event, batch_size=len(X_val_event_c))
        val_data_f_combs_resource = TestData(torch.FloatTensor(X_val_resource_c))
        val_loader_f_combs_resource = DataLoader(dataset=val_data_f_combs_resource, batch_size=len(X_val_resource_c))
        val_data_f_combs_m = TestData(torch.FloatTensor(X_val_m_c))
        val_loader_f_combs_m = DataLoader(dataset=val_data_f_combs_m, batch_size=len(X_val_m_c))
        val_data_f_combs_TA = TestData(torch.FloatTensor(X_val_TA_c))
        val_loader_f_combs_TA = DataLoader(dataset=val_data_f_combs_TA, batch_size=len(X_val_TA_c))

        y_output_val = []
        with torch.no_grad():
            steps_r = list((val_loader_f_combs_resource))
            steps_ta = list((val_loader_f_combs_TA))
            steps_m = list((val_loader_f_combs_m))
            for i, X_batch in enumerate(val_loader_f_combs_event):
                y_test_pred = model(X_batch.to(torch.int64).to(device), steps_r[i].to(torch.int64).to(device),
                                    steps_ta[i].to(torch.float).to(device), steps_m[i].to(torch.int64).to(device))
                
                # change
                y_test_pred = y_test_pred.cpu()
                
                y_output_val.append(y_test_pred.numpy())

        outputs_train['NoDev' + str(d)] = y_output_train[0][:, 0]
        outputs_train['Dev' + str(d)] = y_output_train[0][:, 1]
        outputs_test['NoDev' + str(d)] = y_output_test[0][:, 0]
        outputs_test['Dev' + str(d)] = y_output_test[0][:, 1]
        outputs_val['NoDev' + str(d)] = y_output_val[0][:, 0]
        outputs_val['Dev' + str(d)] = y_output_val[0][:, 1]
        if d == dev[0]:
            outputs_train['prefix_length'] = flatten_comprehension(pref_list_train_c)
            outputs_test['prefix_length'] = flatten_comprehension(pref_list_test_c)
            outputs_val['prefix_length'] = flatten_comprehension(pref_list_val_c)


    # metrics.to_excel(writer, sheet_name=('Metrics'))
    # writer.close()

    # ------------------------------------------------------------------
    # NEW: print unweighted macro avg precision/recall over Dev + NoDev
    # ------------------------------------------------------------------
    try:
        dev_prec, dev_rec = [], []
        nodev_prec, nodev_rec = [], []
        for d in dev:
            try:
                p_dev = float(metrics[d]['Precision'])
                r_dev = float(metrics[d]['Recall'])
                p_nd = float(metrics[str('NoDev' + d)]['Precision'])
                r_nd = float(metrics[str('NoDev' + d)]['Recall'])
            except Exception:
                continue
            if np.isfinite(p_dev) and np.isfinite(r_dev):
                dev_prec.append(p_dev); dev_rec.append(r_dev)
            if np.isfinite(p_nd) and np.isfinite(r_nd):
                nodev_prec.append(p_nd); nodev_rec.append(r_nd)

        macro_p_dev = float(np.mean(dev_prec)) if len(dev_prec) else float("nan")
        macro_r_dev = float(np.mean(dev_rec)) if len(dev_rec) else float("nan")
        macro_p_nd  = float(np.mean(nodev_prec)) if len(nodev_prec) else float("nan")
        macro_r_nd  = float(np.mean(nodev_rec)) if len(nodev_rec) else float("nan")

        print("\n=== Unweighted macro averages across deviation types ===")
        print(f"Dev   : Precision={macro_p_dev:.4f} | Recall={macro_r_dev:.4f}")
        print(f"NoDev : Precision={macro_p_nd:.4f} | Recall={macro_r_nd:.4f}\n")
    except Exception as er:
        print("Macro-average print failed:", er)

In [ ]:
# LSTM seperate
IDP_separate_LSTM_CIBE(log, ref_log, aligned_traces)

In [ ]:
def IDP_collective_LSTM_CIBE(log, ref_log, aligned_traces, u_sample = True,early_stop = True,explained = False,split = 1 / 3):
    xt, z = os.path.split(file)
    import warnings
    warnings.simplefilter('ignore')
    #### get information whether deviation happened after prefix length in DF
    i = 0
    dev = []  # stores all deviations that happened
    for trace in log:
        no_moves = len(aligned_traces[i]['alignment'])
        for j in range(0, len(aligned_traces[i]['alignment'])):
            if aligned_traces[i]['alignment'][j][1] == None or aligned_traces[i]['alignment'][j][0] == \
                    aligned_traces[i]['alignment'][j][1]:
                next  # we do not care for simultaneous or silent moves
            else:
                if not str(aligned_traces[i]['alignment'][j]) in dev:
                    dev.append(str(aligned_traces[i]['alignment'][j]))
        i += 1

    y_cum_test = {}  # dict that stores label for each prefix and deviation combinations; keys are prefix length, entries are Data Frames with index = trace and columns = deviation
    dev_df = pd.DataFrame(data=0, columns=dev, index=range(
        len(log)))  # Data Frame that stores the information whether a deviation happened for each trace on trace level
    event_order = {}  # dict with event sequences for each trace
    event_count = {}  # dict with trace length for each trace
    max_ev = 0  # will be maximum trace length
    k = 0
    for trace in log:
        event_order[k] = []
        i = 0
        for event in trace:
            i += 1
            event_order[k].append(event['concept:name'])
        if i > max_ev:
            max_ev = i
        event_count[k] = len(event_order[k])
        k += 1
    i = 0
    for trace in log:
        no_moves = len(aligned_traces[i]['alignment'])
        for j in range(0, len(aligned_traces[i]['alignment'])):
            if aligned_traces[i]['alignment'][j][1] == None or aligned_traces[i]['alignment'][j][0] == \
                    aligned_traces[i]['alignment'][j][1]:
                next
            else:
                dev_df[str(aligned_traces[i]['alignment'][j])][i] = 1
        i += 1
    for ev in range(1, max_ev + 1):
        y_cum_test[
            ev] = dev_df.copy()  # initialize each prefix length with all traces and information whether deviation happened
    for ev in range(1, max_ev + 1):
        drop_idx = []
        for trace_idx in range(len(log)):
            if event_count[trace_idx] < ev:
                drop_idx.append(trace_idx)  # drop all trace labels that do not go until prefix length
        y_cum_test[ev] = y_cum_test[ev].drop(drop_idx)
    i = 0
    for trace in log:
        no_moves = len(aligned_traces[i]['alignment'])
        j = no_moves - 1  # iterator over moves in alignment, starting at the end
        m = len(event_order[i])  # iterator over event sequence, starting at the end
        while j >= 0:
            if aligned_traces[i]['alignment'][j][
                1] == None:  # if silent move, just go one move further to the beginning in the alignment
                j -= 1
            elif aligned_traces[i]['alignment'][j][0] == aligned_traces[i]['alignment'][j][1]:
                if event_order[i][m - 1] == aligned_traces[i]['alignment'][j][
                    0]:  # if synchronous move, just go one move further to the beginning in the alignment and one event forther to the beginning in the event sequence
                    j -= 1
                    m -= 1
            elif event_order[i][m - 1] == aligned_traces[i]['alignment'][j][0]:  # log move detected
                for q in range(m, max_ev + 1):
                    y_cum_test[q][str(aligned_traces[i]['alignment'][j])][
                        i] = 0  # set all prefixes from the current m to the maximum prefix length in this trace to 0 because deviation happened here but not afterwards
                j -= 1
                m -= 1
            elif m == max_ev:
                j -= 1
            else:  # model move deteceted
                for q in range(m + 1, max_ev + 1):
                    y_cum_test[q][str(aligned_traces[i]['alignment'][j])][
                        i] = 0  # set all prefixes after the current m to the maximum prefix length in this trace to 0 because deviation happened between m and m+1 but not afterwards
                j -= 1
        i += 1
    ### y_cum_test holds information whether deviation happened after prefix length
    x_train_idx, x_test_idx, y_train_idx, y_test_idx = train_test_split(range(len(log)), range(len(log)), test_size=split,
                                                                        random_state=0)
    trainin_dev_df = dev_df.loc[x_train_idx]

    trainin_dev_df.corr()
    corrMatrix = trainin_dev_df.corr()

    corrMatrix.loc[:, :] = np.tril(corrMatrix, k=-1)  # borrowed from Karl D's answer

    already_in = set()
    max_combs_l = []
    for col in corrMatrix:
        # perfect_corr = corrMatrix[col][corrMatrix[col] >= relevance_ths].index.tolist()
        # FIX: relevance_ths is not in the signature of this function; use a safe default
        relevance_ths = 0.5
        perfect_corr = corrMatrix[col][corrMatrix[col] >= relevance_ths].index.tolist()
        if perfect_corr and col not in already_in:
            already_in.update(set(perfect_corr))
            perfect_corr.append(col)
            max_combs_l.append(perfect_corr)

    test_counts = {}
    for comb in max_combs_l:
        for y in range(len(comb)):
            test_counts[comb[y]] = dev_df.loc[x_test_idx].sum()[comb[y]]
        if any(dev_df.loc[x_test_idx].sum()[comb[y]] == 0 for y in range(len(comb))):
            max_combs_l.remove(comb)
            print(comb)

    max_combs = {}
    for comb in max_combs_l:
        max_combs[str(comb)] = comb

    y_cum_test_combs = {}
    for prefix in range(1, max_ev + 1):
        y_cum_test_combs[prefix] = y_cum_test[prefix].copy(deep=True)
        for comb in max_combs.keys():
            y_cum_test_combs[prefix][comb] = 0
            for i in list(y_cum_test_combs[prefix].index):
                if event_count[i] < prefix:
                    continue
                if all(y_cum_test_combs[prefix][j][i] == 1 for j in max_combs[comb]):
                    for j in max_combs[comb]:
                        y_cum_test_combs[prefix][j][i] = 0
                    y_cum_test_combs[prefix][comb][i] = 1
    trainin_dev_df.sum()
    y_cum_test_o_combs = {}
    for prefix in range(1, max_ev + 1):
        y_cum_test_o_combs[prefix] = y_cum_test_combs[prefix][list(max_combs.keys())]
    y_cum_test_o_combs[1].sum()

    ## ref_log will have all attributes that will be the columns for X_test and X_train
    ref_log = complex_index_encoding(ref_log,
                                     4000)  # prepare a log with the maximum length of the feature vector from CIBE to know to pad other feature vectors
    ref_dataframe1 = pm4py.convert_to_dataframe(ref_log)
    ref_dataframe = ref_dataframe1.drop_duplicates(subset=['case:concept:name'])
    ref_dataframe = ref_dataframe.filter(like='case:', axis=1)
    ref_dataframe = ref_dataframe.drop('case:concept:name', axis=1)
    ref_dataframe.columns = ref_dataframe.columns.str.replace('case:', '')
    ref_dataframe = ref_dataframe.reset_index()
    ref_raw_dat = ref_dataframe.drop('index', axis=1)
    ## dataset-specific preparation (i.e., redundant attributes, convertion to numeric)
    if z == 'aligned_traces_12A.pkl' or z == 'aligned_traces_12O.pkl' or z == 'aligned_traces_12AO.pkl':
        ref_raw_dat['AMOUNT_REQ'] = pd.to_numeric(ref_raw_dat['AMOUNT_REQ'])
        ref_clean_dat = ref_raw_dat.drop('REG_DATE', axis=1)
    elif z == 'aligned_traces_20int.pkl':
        ref_clean_dat = ref_raw_dat.drop(
            ['Permit travel permit number', 'DeclarationNumber', 'travel permit number', 'id', 'Permit ID', 'Permit id'],
            axis=1)
    elif z == 'aligned_traces_20dom.pkl':
        ref_clean_dat = ref_raw_dat.drop(['DeclarationNumber', 'id'], axis=1)
    elif z == 'aligned_traces_20prep.pkl':
        ref_clean_dat = ref_raw_dat.drop(['RfpNumber', 'Rfp_id', 'Permit travel permit number', 'Permit id'], axis=1)
    elif z == 'aligned_traces_20RfP.pkl':
        ref_clean_dat = ref_raw_dat.drop(['RfpNumber', 'Rfp_id'], axis=1)
    else:
        ref_clean_dat = ref_raw_dat.copy()

    ref_enc_dat = ref_clean_dat.copy()

    BATCH_SIZE = 128
    LEARNING_RATE = 0.00001

    X_cum = {}
    metrics = pd.DataFrame(data=0, columns=dev, index=['Precision', 'Recall', 'Support', 'ROC_AUC'])

    # prepare X for all prefix lengths
    for prefix in range(1, max_ev + 1):
        complex_index_encoding(log, prefix)
        dataframe1 = pm4py.convert_to_dataframe(log)
        dataframe = dataframe1.drop_duplicates(subset=['case:concept:name'])
        dataframe = dataframe.filter(like='case:', axis=1)
        dataframe = dataframe.drop('case:concept:name', axis=1)
        dataframe.columns = dataframe.columns.str.replace('case:', '')
        dataframe = dataframe.reset_index()
        raw_dat = dataframe.drop('index', axis=1)
        if z == 'aligned_traces_12A.pkl' or z == 'aligned_traces_12O.pkl' or z == 'aligned_traces_12AO.pkl':
            raw_dat['AMOUNT_REQ'] = pd.to_numeric(raw_dat['AMOUNT_REQ'])
            clean_dat = raw_dat.drop('REG_DATE', axis=1)
        elif z == 'aligned_traces_20int.pkl':
            clean_dat = raw_dat.drop(
                ['Permit travel permit number', 'DeclarationNumber', 'travel permit number', 'id', 'Permit ID',
                 'Permit id'], axis=1)
        elif z == 'aligned_traces_20dom.pkl':
            clean_dat = raw_dat.drop(['DeclarationNumber', 'id'], axis=1)
        elif z == 'aligned_traces_20prep.pkl':
            clean_dat = raw_dat.drop(['RfpNumber', 'Rfp_id', 'Permit travel permit number', 'Permit id'], axis=1)
        elif z == 'aligned_traces_20RfP.pkl':
            clean_dat = raw_dat.drop(['RfpNumber', 'Rfp_id'], axis=1)
        else:
            clean_dat = raw_dat.copy()
        enc_dat = clean_dat.copy()
        for key in ref_enc_dat.columns:
            if not key in enc_dat.columns:
                enc_dat[key] = 'No'  # pad all prefixes to maximum lengths with 0
        imp = SimpleImputer(missing_values=np.nan, strategy='constant', fill_value='No')
        enc_dat = pd.DataFrame(data=imp.fit_transform(enc_dat), columns=enc_dat.columns)

        X_cum[prefix] = enc_dat.copy()
        drop_idx = []
        for trace_idx in range(len(log)):
            if event_count[trace_idx] < prefix:
                drop_idx.append(trace_idx)

        X_cum[prefix] = X_cum[prefix].drop(drop_idx)

    for d in dev:
        metrics[str('NoDev' + d)] = 0

    path = (os.getcwd() + '/BPDP_LSTM')  # output path
    xt, z = os.path.split(file)

    # writer = pd.ExcelWriter(path + '/' + z + '_BPDP_CIBE_classification.xlsx', engine="xlsxwriter")

    x_train_idx, x_test_idx, y_train_idx, y_test_idx = train_test_split(range(len(log)), range(len(log)), test_size=split,
                                                                        random_state=0)

    x_train_idx_c, x_test_idx_c, y_train_idx_c, y_test_idx_c = train_test_split(range(len(log)), range(len(log)),
                                                                                test_size=split,
                                                                                random_state=0)
    x_train_idx_c, x_val_idx_c, y_train_idx_c, y_val_idx_c = train_test_split(x_train_idx_c, x_train_idx_c, test_size=0.2,
                                                                              random_state=0)

    dev_position = pd.DataFrame(index=x_test_idx, columns=dev, data=0)
    for d in dev:
        for idx in x_test_idx:
            for i in range(1, event_count[idx] + 1):
                if y_cum_test[i][d][idx] == 1: dev_position[d][idx] = i + 1
    dev_position_pred = pd.DataFrame(index=x_test_idx, columns=dev, data=0)
    earliness = {}

    dev_distribution = pd.DataFrame(data=0, index=['Training', 'Test'], columns=dev)
    for d in dev:
        dev_distribution[d]['Training'] = sum(dev_df[d][i] for i in x_train_idx)
        dev_distribution[d]['Test'] = sum(dev_df[d][i] for i in x_test_idx)

    # dev_distribution.to_excel(writer, sheet_name=('Distribution'))


    def flatten_comprehension(matrix):
        return [item for row in matrix for item in row]


    ref_enc_dat

    evs_c = []
    resource_c = []
    month_c = []
    trace_attr = []
    for ca in X_cum[1].columns:
        if ca.startswith('event'): evs_c.append(ca)
    for ca in X_cum[1].columns:
        if ca.startswith('resource'): resource_c.append(ca)
    for ca in X_cum[1].columns:
        if ca.startswith('month'): month_c.append(ca)
    for ca in X_cum[1].columns:
        if not (ca in evs_c or ca in resource_c or ca in month_c): trace_attr.append(ca)
    print(evs_c)
    print(resource_c)
    print(month_c)
    print(trace_attr)

    X_events = {}
    X_resource = {}
    X_month = {}
    X_tracea = {}
    for prefix in range(1, max_ev + 1):
        X_events[prefix] = X_cum[prefix][evs_c]
        X_resource[prefix] = X_cum[prefix][resource_c]
        X_month[prefix] = X_cum[prefix][month_c]
        X_tracea[prefix] = X_cum[prefix][trace_attr]

    cat_tas = []
    for cat in X_tracea[1].columns:
        if type(X_tracea[1][cat][0]) == str:
            cat_tas.append(cat)
    cat_tas
    uniques_cats = {}
    for cat in cat_tas:
        uniques_cats[cat] = []
    for prefix in range(1, max_ev + 1):
        for cat in cat_tas:
            for reals in list(X_tracea[prefix][cat].unique()):
                if not reals in uniques_cats[cat]:
                    uniques_cats[cat].append(reals)
    uniques_cats
    X_tracea[1]
    for cat in cat_tas:
        for prefix in range(1, max_ev + 1):
            for j in list(X_tracea[prefix].index):
                X_tracea[prefix][cat][j] = uniques_cats[cat].index(X_tracea[prefix][cat][j])
    X_tracea[1]
    X_events[1]
    positive_weights = {}
    negative_weights = {}
    for label in dev:
        positive_weights[label] = 8
        negative_weights[label] = 1
    models_collect = {}
    dev_trained = []
    outputs_train = pd.DataFrame()
    outputs_test = pd.DataFrame()
    outputs_val = pd.DataFrame()

    # Y_cum_dev = {}
    # for prefix in range(1, max_ev + 1):
    #     Y_cum_dev[prefix] = pd.DataFrame(y_cum_test[prefix][d])
    #     Y_cum_dev[prefix]['NoDev'] = 0
    #     for i in Y_cum_dev[prefix].index.values.tolist():
    #         Y_cum_dev[prefix]['NoDev'][i] = 1 - y_cum_test[prefix][d][i]
    #     if prefix == 1:
    #         print(Y_cum_dev[prefix].columns)
    # FIX: collective -> use multi-label y directly (all deviations) and also create a matching "NoDev" matrix
    Y_cum_dev = {}
    for prefix in range(1, max_ev + 1):
        Y_cum_dev[prefix] = y_cum_test[prefix].copy(deep=True)  # columns = dev
        # Create NoDev_* columns for each deviation (for metrics printing later)
        for d in dev:
            Y_cum_dev[prefix]['NoDev_' + d] = 1 - y_cum_test[prefix][d]
        if prefix == 1:
            print(Y_cum_dev[prefix].columns)

    if u_sample:
        imb_ref_enc_dat = pd.get_dummies(ref_clean_dat)
        imb_ref_enc_dat['ind'] = 0
        for i in range(len(imb_ref_enc_dat)):
            imb_ref_enc_dat['ind'][i] = i

        imb_traces = pd.DataFrame(data=0, columns=['Dev'], index=range(len(log)))
        for trace in range(len(log)):
            if dev_df.loc[trace].sum() > 0:
                imb_traces['Dev'][trace] = 1

        imb_traces = imb_traces.drop(x_test_idx)
        imb_ref_enc_dat = imb_ref_enc_dat.drop(x_test_idx)
        imp = SimpleImputer(missing_values=np.nan, strategy='constant', fill_value=0)
        imb_ref_enc_dat = pd.DataFrame(data=imp.fit_transform(imb_ref_enc_dat), columns=imb_ref_enc_dat.columns)

        oss = OneSidedSelection(random_state=0, n_seeds_S=250, n_neighbors=7)

        X_resampled, y_resampled = oss.fit_resample(imb_ref_enc_dat, imb_traces)

        x_train_idx = list(X_resampled['ind'])
        y_train_idx = list(X_resampled['ind'])

    print('index length ', len(x_train_idx), len(x_test_idx), len(y_train_idx), len(y_test_idx))

    # validation set for early stopping
    x_train_idx, x_val_idx, y_train_idx, y_val_idx = train_test_split(x_train_idx, x_train_idx, test_size=0.2,

                                                                      random_state=0)

    enumerated_trace_idx = {}
    cum_trace_idxs = []
    pref_list = []
    pref_list_train_c = []
    pref_list_test_c = []
    pref_list_val_c = []
    for prefix in range(1, max_ev + 1):
        drop_idx = []
        for trace_idx in range(len(log)):
            if event_count[trace_idx] < prefix:
                drop_idx.append(trace_idx)  # drop all trace encoding that do not go until prefix length

        x_te = X_events[prefix].loc[[j for j in list(set(y_test_idx) - set(drop_idx))]].to_numpy()
        x_tr = X_events[prefix].loc[[j for j in list(set(y_train_idx) - set(drop_idx))]].to_numpy()
        x_va = X_events[prefix].loc[[j for j in list(set(y_val_idx) - set(drop_idx))]].to_numpy()

        # y_te = y_cum_test[prefix].loc[[j for j in list(set(y_test_idx) - set(drop_idx))]].to_numpy().astype(float)
        # y_tr = y_cum_test[prefix].loc[[j for j in list(set(y_train_idx) - set(drop_idx))]].to_numpy().astype(float)
        # y_va = y_cum_test[prefix].loc[[j for j in list(set(y_val_idx) - set(drop_idx))]].to_numpy().astype(float)
        # FIX: use multi-label targets from Y_cum_dev
        y_te = Y_cum_dev[prefix].loc[[j for j in list(set(y_test_idx) - set(drop_idx))]][dev].to_numpy().astype(float)
        y_tr = Y_cum_dev[prefix].loc[[j for j in list(set(y_train_idx) - set(drop_idx))]][dev].to_numpy().astype(float)
        y_va = Y_cum_dev[prefix].loc[[j for j in list(set(y_val_idx) - set(drop_idx))]][dev].to_numpy().astype(float)

        x_te_c = X_events[prefix].loc[[j for j in list(set(y_test_idx_c) - set(drop_idx))]].to_numpy()
        x_tr_c = X_events[prefix].loc[[j for j in list(set(y_train_idx_c) - set(drop_idx))]].to_numpy()
        x_va_c = X_events[prefix].loc[[j for j in list(set(y_val_idx_c) - set(drop_idx))]].to_numpy()
        y_te_c = y_cum_test_o_combs[prefix].loc[[j for j in list(set(y_test_idx) - set(drop_idx))]].to_numpy().astype(
            float)
        y_tr_c = y_cum_test_o_combs[prefix].loc[
            [j for j in list(set(y_train_idx_c) - set(drop_idx))]].to_numpy().astype(float)
        y_va_c = y_cum_test_o_combs[prefix].loc[[j for j in list(set(y_val_idx_c) - set(drop_idx))]].to_numpy().astype(
            float)
        cum_trace_idxs.append(list(set(y_test_idx) - set(drop_idx)))
        pref_list.append([prefix] * len(list(set(y_test_idx) - set(drop_idx))))
        enumerated_trace_idx[prefix] = list(set(y_test_idx) - set(drop_idx))
        pref_list_train_c.append([prefix] * len(list(set(y_train_idx_c) - set(drop_idx))))
        pref_list_test_c.append([prefix] * len(list(set(y_test_idx) - set(drop_idx))))
        pref_list_val_c.append([prefix] * len(list(set(y_val_idx_c) - set(drop_idx))))
        print('subset length ', prefix, len(x_te), len(x_tr), len(y_te), len(y_tr))

        if prefix == 1:
            X_train_event = x_tr
            X_test_event = x_te
            y_train = y_tr
            y_test = y_te
            X_val_event = x_va
            y_val = y_va
            X_train_event_c = x_tr_c
            X_test_event_c = x_te_c
            X_val_event_c = x_va_c
            y_train_c = y_tr_c
            y_test_c = y_te_c
            y_val_c = y_va_c
        else:
            X_train_event = np.append(X_train_event, x_tr, axis=0)
            X_test_event = np.append(X_test_event, x_te, axis=0)
            y_train = np.append(y_train, y_tr, axis=0)
            y_test = np.append(y_test, y_te, axis=0)
            y_val = np.append(y_val, y_va, axis=0)
            X_val_event = np.append(X_val_event, x_va, axis=0)
            X_train_event_c = np.append(X_train_event_c, x_tr_c, axis=0)
            X_test_event_c = np.append(X_test_event_c, x_te_c, axis=0)
            X_val_event_c = np.append(X_val_event_c, x_va_c, axis=0)
            y_train_c = np.append(y_train_c, y_tr_c, axis=0)
            y_test_c = np.append(y_test_c, y_te_c, axis=0)
            y_val_c = np.append(y_val_c, y_va_c, axis=0)  # combine all X data from all prefixes into one array
    print("collective", len(X_train_event), len(y_train), len(X_val_event), len(y_val), len(X_test_event), len(y_test))

    for prefix in range(1, max_ev + 1):
        drop_idx = []
        for trace_idx in range(len(log)):
            if event_count[trace_idx] < prefix:
                drop_idx.append(trace_idx)  # drop all trace encoding that do not go until prefix length

        x_te = X_resource[prefix].loc[[j for j in list(set(y_test_idx) - set(drop_idx))]].to_numpy()
        x_tr = X_resource[prefix].loc[[j for j in list(set(y_train_idx) - set(drop_idx))]].to_numpy()
        x_va = X_resource[prefix].loc[[j for j in list(set(y_val_idx) - set(drop_idx))]].to_numpy()
        x_te_c = X_resource[prefix].loc[[j for j in list(set(y_test_idx_c) - set(drop_idx))]].to_numpy()
        x_tr_c = X_resource[prefix].loc[[j for j in list(set(y_train_idx_c) - set(drop_idx))]].to_numpy()
        x_va_c = X_resource[prefix].loc[[j for j in list(set(y_val_idx_c) - set(drop_idx))]].to_numpy()
        print('subset length ', prefix, len(x_te), len(x_tr))

        if prefix == 1:
            X_train_resource = x_tr
            X_test_resource = x_te
            X_val_resource = x_va
            X_train_resource_c = x_tr_c
            X_test_resource_c = x_te_c
            X_val_resource_c = x_va_c
        else:
            X_train_resource = np.append(X_train_resource, x_tr, axis=0)
            X_test_resource = np.append(X_test_resource, x_te, axis=0)
            X_val_resource = np.append(X_val_resource, x_va, axis=0)
            X_train_resource_c = np.append(X_train_resource_c, x_tr_c, axis=0)
            X_test_resource_c = np.append(X_test_resource_c, x_te_c, axis=0)
            X_val_resource_c = np.append(X_val_resource_c, x_va_c,
                                         axis=0)  # combine all X data from all prefixes into one array

    for prefix in range(1, max_ev + 1):
        drop_idx = []
        for trace_idx in range(len(log)):
            if event_count[trace_idx] < prefix:
                drop_idx.append(trace_idx)  # drop all trace encoding that do not go until prefix length

        x_te = X_month[prefix].loc[[j for j in list(set(y_test_idx) - set(drop_idx))]].to_numpy()
        x_tr = X_month[prefix].loc[[j for j in list(set(y_train_idx) - set(drop_idx))]].to_numpy()
        x_va = X_month[prefix].loc[[j for j in list(set(y_val_idx) - set(drop_idx))]].to_numpy()
        x_te_c = X_month[prefix].loc[[j for j in list(set(y_test_idx_c) - set(drop_idx))]].to_numpy()
        x_tr_c = X_month[prefix].loc[[j for j in list(set(y_train_idx_c) - set(drop_idx))]].to_numpy()
        x_va_c = X_month[prefix].loc[[j for j in list(set(y_val_idx_c) - set(drop_idx))]].to_numpy()
        print('subset length ', prefix, len(x_te), len(x_tr))

        if prefix == 1:
            X_train_m = x_tr
            X_test_m = x_te
            X_val_m = x_va
            X_train_m_c = x_tr_c
            X_test_m_c = x_te_c
            X_val_m_c = x_va_c
        else:
            X_train_m = np.append(X_train_m, x_tr, axis=0)
            X_test_m = np.append(X_test_m, x_te, axis=0)
            X_val_m = np.append(X_val_m, x_va, axis=0)
            X_train_m_c = np.append(X_train_m_c, x_tr_c, axis=0)
            X_test_m_c = np.append(X_test_m_c, x_te_c, axis=0)
            X_val_m_c = np.append(X_val_m_c, x_va_c, axis=0)  # combine all X data from all prefixes into one array

    for prefix in range(1, max_ev + 1):
        drop_idx = []
        for trace_idx in range(len(log)):
            if event_count[trace_idx] < prefix:
                drop_idx.append(trace_idx)  # drop all trace encoding that do not go until prefix length

        x_te = X_tracea[prefix].loc[[j for j in list(set(y_test_idx) - set(drop_idx))]].to_numpy()
        x_tr = X_tracea[prefix].loc[[j for j in list(set(y_train_idx) - set(drop_idx))]].to_numpy()
        x_va = X_tracea[prefix].loc[[j for j in list(set(y_val_idx) - set(drop_idx))]].to_numpy()
        x_te_c = X_tracea[prefix].loc[[j for j in list(set(y_test_idx_c) - set(drop_idx))]].to_numpy()
        x_tr_c = X_tracea[prefix].loc[[j for j in list(set(y_train_idx_c) - set(drop_idx))]].to_numpy()
        x_va_c = X_tracea[prefix].loc[[j for j in list(set(y_val_idx_c) - set(drop_idx))]].to_numpy()
        print('subset length ', prefix, len(x_te), len(x_tr))

        if prefix == 1:
            X_train_TA = x_tr
            X_test_TA = x_te
            X_val_TA = x_va
            X_train_TA_c = x_tr_c
            X_test_TA_c = x_te_c
            X_val_TA_c = x_va_c
        else:
            X_train_TA = np.append(X_train_TA, x_tr, axis=0)
            X_test_TA = np.append(X_test_TA, x_te, axis=0)
            X_val_TA = np.append(X_val_TA, x_va, axis=0)
            X_train_TA_c = np.append(X_train_TA_c, x_tr_c, axis=0)
            X_test_TA_c = np.append(X_test_TA_c, x_te_c, axis=0)
            X_val_TA_c = np.append(X_val_TA_c, x_va_c, axis=0)  # combine all X data from all prefixes into one array

    events_encoder = list(
        np.unique(np.append(np.append(X_train_event_c, X_test_event_c, axis=0), X_val_event_c, axis=0)))
    events_encoder.index('No')
    resource_encoder = list(
        np.unique(np.append(np.append(X_train_resource_c, X_test_resource_c, axis=0), X_val_resource_c, axis=0)))
    resource_encoder.index('No')

    month_encoder = list(np.unique(np.append(np.append(X_train_m_c, X_test_m_c, axis=0), X_val_m_c, axis=0)))

    for i in range(len(X_test_event)):
        for j in range(len(X_test_event[0])):
            X_test_event[i][j] = events_encoder.index(X_test_event[i][j])
        for j in range(len(X_test_resource[0])):
            X_test_resource[i][j] = resource_encoder.index(X_test_resource[i][j])
        for j in range(len(X_test_m[0])):
            X_test_m[i][j] = month_encoder.index(X_test_m[i][j])
    for i in range(len(X_train_event)):
        for j in range(len(X_train_event[0])):
            X_train_event[i][j] = events_encoder.index(X_train_event[i][j])
        for j in range(len(X_train_resource[0])):
            X_train_resource[i][j] = resource_encoder.index(X_train_resource[i][j])
        for j in range(len(X_train_m[0])):
            X_train_m[i][j] = month_encoder.index(X_train_m[i][j])
    for i in range(len(X_val_event)):
        for j in range(len(X_val_event[0])):
            X_val_event[i][j] = events_encoder.index(X_val_event[i][j])
        for j in range(len(X_val_resource[0])):
            X_val_resource[i][j] = resource_encoder.index(X_val_resource[i][j])
        for j in range(len(X_val_m[0])):
            X_val_m[i][j] = month_encoder.index(X_val_m[i][j])

    # combs_output encoders as well
    for i in range(len(X_test_event_c)):
        for j in range(len(X_test_event_c[0])):
            X_test_event_c[i][j] = events_encoder.index(X_test_event_c[i][j])
        for j in range(len(X_test_resource_c[0])):
            X_test_resource_c[i][j] = resource_encoder.index(X_test_resource_c[i][j])
        for j in range(len(X_test_m_c[0])):
            X_test_m_c[i][j] = month_encoder.index(X_test_m_c[i][j])
    for i in range(len(X_train_event_c)):
        for j in range(len(X_train_event_c[0])):
            X_train_event_c[i][j] = events_encoder.index(X_train_event_c[i][j])
        for j in range(len(X_train_resource_c[0])):
            X_train_resource_c[i][j] = resource_encoder.index(X_train_resource_c[i][j])
        for j in range(len(X_train_m_c[0])):
            X_train_m_c[i][j] = month_encoder.index(X_train_m_c[i][j])
    for i in range(len(X_val_event_c)):
        for j in range(len(X_val_event_c[0])):
            X_val_event_c[i][j] = events_encoder.index(X_val_event_c[i][j])
        for j in range(len(X_val_resource_c[0])):
            X_val_resource_c[i][j] = resource_encoder.index(X_val_resource_c[i][j])
        for j in range(len(X_val_m_c[0])):
            X_val_m_c[i][j] = month_encoder.index(X_val_m_c[i][j])

    scaler = StandardScaler()
    X_test_TA = scaler.fit_transform(X_test_TA)
    X_train_TA = scaler.fit_transform(X_train_TA)
    X_val_TA = scaler.fit_transform(X_val_TA)

    X_test_event = X_test_event.astype(int)
    X_train_event = X_train_event.astype(int)
    X_val_event = X_val_event.astype(int)
    X_test_resource = X_test_resource.astype(int)
    X_train_resource = X_train_resource.astype(int)
    X_val_resource = X_val_resource.astype(int)
    X_test_m = X_test_m.astype(int)
    X_train_m = X_train_m.astype(int)
    X_val_m = X_val_m.astype(int)

    X_test_event_c = X_test_event_c.astype(int)
    X_train_event_c = X_train_event_c.astype(int)
    X_val_event_c = X_val_event_c.astype(int)
    X_test_resource_c = X_test_resource_c.astype(int)
    X_train_resource_c = X_train_resource_c.astype(int)
    X_val_resource_c = X_val_resource_c.astype(int)
    X_test_m_c = X_test_m_c.astype(int)
    X_train_m_c = X_train_m_c.astype(int)
    X_val_m_c = X_val_m_c.astype(int)

    class BPDP_LSTM_SC(nn.Module):
        def __init__(self, vocab_events, vocab_resources, no_TA, vocab_month, no_devs):
            super(BPDP_LSTM_SC, self).__init__()
            self.embedding_e = nn.Embedding(vocab_events, 16)  # hier auf 8 / 16
            self.activation1 = nn.LeakyReLU()
            self.lstm_e = nn.LSTM(input_size=16, hidden_size=64, num_layers=1, batch_first=True, dropout=0.1)
            self.linear_e = nn.Linear(64, 32)
            self.embedding_r = nn.Embedding(vocab_resources, 16)
            self.lstm_r = nn.LSTM(input_size=16, hidden_size=64, num_layers=1, batch_first=True, dropout=0.1)
            self.linear_r = nn.Linear(64, 32)
            self.embedding_m = nn.Embedding(vocab_month, 16)
            self.lstm_m = nn.LSTM(input_size=16, hidden_size=64, num_layers=1, batch_first=True, dropout=0.1)
            self.linear_m = nn.Linear(64, 32)
            self.linear_ta = nn.Linear(no_TA, 32)
            self.dropout = nn.Dropout(p=0.1)
            self.batchnorm1 = nn.LayerNorm(128)
            self.linear = nn.Linear(128, no_devs)

        def forward(self, evs, rs, tas, ms):
            evs = self.embedding_e(evs)
            evs, _ = self.lstm_e(evs)
            evs = self.linear_e(evs)
            evs = evs[:, -1, :]
            evs = self.activation1(evs)
            rs = self.embedding_r(rs)
            rs, _ = self.lstm_r(rs)
            rs = rs[:, -1, :]
            rs = self.activation1(rs)
            rs = self.linear_r(rs)
            ms = self.embedding_m(ms)
            ms, _ = self.lstm_m(ms)
            ms = ms[:, -1, :]
            ms = self.activation1(ms)
            ms = self.linear_m(ms)
            tas = self.linear_ta(tas)
            fin = torch.cat((evs, rs), dim=1)
            fin = torch.cat((fin, ms), dim=1)
            fin = torch.cat((fin, tas), dim=1)
            fin = self.batchnorm1(fin)
            #fin = self.dropout(fin)
            fin = self.linear(fin)
            return fin


    labels = dev  # ['label_1', ...., 'label_6']

    positives = {}
    negatives = {}
    for label in labels:
        positives[label] = sum(dev_df[label] == 1)
        negatives[label] = sum(dev_df[label] == 0)
    max_Plabel = max(positives.values())
    max_Nlabel = max(negatives.values())
    max_label = max(max_Plabel, max_Nlabel)
    pir = {}
    nir = {}
    pirlbl = {}
    nirlbl = {}
    for label in labels:
        pir[label] = max(positives[label], negatives[label]) / positives[label]
        nir[label] = max(positives[label], negatives[label]) / negatives[label]
        pirlbl[label] = max_label / positives[label]
        nirlbl[label] = max_label / negatives[label]
    positive_weights = {}
    negative_weights = {}
    for label in labels:
        positive_weights[label] = mean(pir.values()) ** (1 / (2 * math.e)) + np.log(pirlbl[label])
        negative_weights[label] = mean(nir.values()) ** (1 / (2 * math.e)) + np.log(nirlbl[label])
    positive_weights

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = BPDP_LSTM_SC(vocab_events=len(events_encoder), vocab_resources=len(resource_encoder), no_TA=len(X_train_TA[0]),
                         vocab_month=len(month_encoder), no_devs=len(y_train[0]))
    model.to(device)

    # weights = torch.FloatTensor(list(positive_weights.values()))
    # criterion = nn.CrossEntropyLoss(weight=weights)
    # FIX: move weights to device; also this is multi-label -> use BCEWithLogitsLoss (NOT CrossEntropyLoss)
    weights = torch.tensor(list(positive_weights.values()), dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=weights)

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    torch.autograd.set_detect_anomaly(True)
    if early_stop:
        EPOCHS = 100
        model.train()
        train_data_event = TrainData(torch.FloatTensor(X_train_event),
                                     torch.FloatTensor(y_train))
        train_data_resource = TestData(torch.FloatTensor(X_train_resource))
        train_data_TA = TestData(torch.FloatTensor(X_train_TA))
        train_data_m = TestData(torch.FloatTensor(X_train_m))
        train_loader_event = DataLoader(dataset=train_data_event, batch_size=BATCH_SIZE, shuffle=False)
        train_loader_resource = DataLoader(dataset=train_data_resource, batch_size=BATCH_SIZE, shuffle=False)
        train_loader_TA = DataLoader(dataset=train_data_TA, batch_size=BATCH_SIZE, shuffle=False)
        train_loader_m = DataLoader(dataset=train_data_m, batch_size=BATCH_SIZE, shuffle=False)

        # NEW: validation tensors on device
        X_val_event_t = torch.tensor(X_val_event, dtype=torch.int64, device=device)
        X_val_resource_t = torch.tensor(X_val_resource, dtype=torch.int64, device=device)
        X_val_TA_t = torch.tensor(X_val_TA, dtype=torch.float32, device=device)
        X_val_m_t = torch.tensor(X_val_m, dtype=torch.int64, device=device)
        y_val_t = torch.tensor(y_val, dtype=torch.float32, device=device)

        es = EarlyStopping()
        done = False

        epoch = 0
        while epoch < EPOCHS and not done:
            epoch += 1
            steps = list(enumerate(train_loader_event))
            pbar = tqdm.tqdm(steps)
            steps_r = list((train_loader_resource))
            steps_ta = list((train_loader_TA))
            steps_m = list((train_loader_m))
            model.train()
            epoch_acc = 0
            epoch_loss = 0
            for i, (x_batch, y_batch) in pbar:
                optimizer.zero_grad()
                y_batch_pred = model(x_batch.to(torch.int64).to(device), steps_r[i].to(torch.int64).to(device),
                                     steps_ta[i].to(torch.float).to(device), steps_m[i].to(torch.int64).to(device))

                # loss = criterion(y_batch_pred, y_batch.to(device))
                # FIX: BCE expects float targets on same device
                y_batch_t = y_batch.to(device).float()
                loss = criterion(y_batch_pred, y_batch_t)

                # acc = binary_acc(y_batch_pred, y_batch.to(device)) / len(y_batch_pred[0])
                # FIX: multi-label accuracy (threshold 0.5 on sigmoid)
                probs = torch.sigmoid(y_batch_pred)
                preds = (probs >= 0.5).float()
                acc = (preds == y_batch_t).float().mean()

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                optimizer.step()
                epoch_acc += acc.item()

                loss, current = loss.item(), (i + 1) * len(x_batch)
                epoch_loss += loss
                if i == len(steps) - 1:
                    model.eval()
                    pred = model(X_val_event_t, X_val_resource_t, X_val_TA_t, X_val_m_t)

                    # vloss = criterion(pred, torch.FloatTensor(y_val))
                    # FIX: validation loss on device
                    vloss = criterion(pred, y_val_t)

                    if es(model, vloss): done = True
                    pbar.set_description(
                        f"Epoch: {epoch}, tloss: {epoch_loss / len(train_loader_event)}, Acc: {epoch_acc / len(train_loader_event):.3f}, vloss: {vloss:>7f}, EStop:[{es.status}]")
                else:
                    pbar.set_description(
                        f"Epoch: {epoch}, tloss {epoch_loss / len(train_loader_event):}, Acc: {epoch_acc / len(train_loader_event):.3f}")

    y_batch_pred
    model.eval()
    test_data_event = TestData(torch.FloatTensor(X_test_event))
    test_data_resource = TestData(torch.FloatTensor(X_test_resource))
    test_data_TA = TestData(torch.FloatTensor(X_test_TA))
    test_data_m = TestData(torch.FloatTensor(X_test_m))
    test_loader_event = DataLoader(dataset=test_data_event, batch_size=1)
    test_loader_resource = DataLoader(dataset=test_data_resource, batch_size=1)
    test_loader_TA = DataLoader(dataset=test_data_TA, batch_size=1)
    test_loader_m = DataLoader(dataset=test_data_m, batch_size=1)
    iterations_r = iter(test_loader_resource)
    iterations_ta = iter(test_loader_TA)
    iterations_m = iter(test_loader_m)
    y_pred_list = []

    with torch.no_grad():
        for i, X_batch in enumerate(test_loader_event):
            X_batch = X_batch.to(device).to(torch.int64)

            # y_test_pred = torch.nn.functional.sigmoid(
            #     model(X_batch, next(iterations_r).to(torch.int64), next(iterations_ta).to(torch.float),
            #           next(iterations_m).to(torch.int64)))
            # y_pred_tag = torch.round(y_test_pred)
            # FIX: ensure aux batches on device + sigmoid dim handling
            logits = model(
                X_batch,
                next(iterations_r).to(torch.int64).to(device),
                next(iterations_ta).to(torch.float).to(device),
                next(iterations_m).to(torch.int64).to(device),
            )
            y_test_pred = torch.sigmoid(logits)
            y_pred_tag = (y_test_pred >= 0.5).float()
            y_pred_list.append(y_pred_tag.cpu().numpy())

    y_pred_list = [a.squeeze().tolist() for a in y_pred_list]

    CM = sklearn.metrics.multilabel_confusion_matrix(y_test, y_pred_list)

    for i, d in enumerate(dev):
        metrics[d]['Precision'] = CM[i][1][1] / (CM[i][1][1] + CM[i][0][1])
        metrics[d]['Recall'] = CM[i][1][1] / (CM[i][1][1] + CM[i][1][0])
        metrics[d]['Support'] = (CM[i][1][1] + CM[i][1][0])
        try:
            metrics[d]['ROC_AUC'] = sklearn.metrics.roc_auc_score(y_test[:, i], np.array(y_pred_list)[:, i],
                                                                  average='macro')
        except Exception as er:
            metrics[d]['ROC_AUC'] = er
        metrics[str('NoDev' + d)]['Precision'] = CM[i][0][0] / (CM[i][0][0] + CM[i][1][0])
        metrics[str('NoDev' + d)]['Recall'] = CM[i][0][0] / (CM[i][0][0] + CM[i][0][1])
        metrics[str('NoDev' + d)]['Support'] = CM[i][0][0] + CM[i][0][1]
    print(CM)

    print(metrics)
    # writer = pd.ExcelWriter('BPDP_LSTM/' + z + '_BPDP_LSTM_SC_1.xlsx', engine="xlsxwriter")
    # metrics.to_excel(writer, sheet_name=('Metrics'))

    # writer.close()

    # ------------------------------------------------------------------
    # NEW: print unweighted macro avg precision/recall for Dev and NoDev
    # ------------------------------------------------------------------
    try:
        dev_prec, dev_rec = [], []
        nodev_prec, nodev_rec = [], []
        for d in dev:
            try:
                p_dev = float(metrics[d]['Precision'])
                r_dev = float(metrics[d]['Recall'])
                p_nd = float(metrics[str('NoDev' + d)]['Precision'])
                r_nd = float(metrics[str('NoDev' + d)]['Recall'])
            except Exception:
                continue
            if np.isfinite(p_dev) and np.isfinite(r_dev):
                dev_prec.append(p_dev); dev_rec.append(r_dev)
            if np.isfinite(p_nd) and np.isfinite(r_nd):
                nodev_prec.append(p_nd); nodev_rec.append(r_nd)

        macro_p_dev = float(np.mean(dev_prec)) if len(dev_prec) else float("nan")
        macro_r_dev = float(np.mean(dev_rec)) if len(dev_rec) else float("nan")
        macro_p_nd  = float(np.mean(nodev_prec)) if len(nodev_prec) else float("nan")
        macro_r_nd  = float(np.mean(nodev_rec)) if len(nodev_rec) else float("nan")

        print("\n=== Unweighted macro averages across deviation types ===")
        print(f"Dev   : Precision={macro_p_dev:.4f} | Recall={macro_r_dev:.4f}")
        print(f"NoDev : Precision={macro_p_nd:.4f} | Recall={macro_r_nd:.4f}\n")
    except Exception as er:
        print("Macro-average print failed:", er)

In [ ]:
# LSTM collective
IDP_collective_LSTM_CIBE(log, ref_log, aligned_traces, u_sample = True,early_stop = True,explained = False,split = 1 / 3)

In [ ]:
##### Here start the comparisons to other design choices